# Aeroelastic QRC on Aquila — corrected hardware verification notebook

This corrected notebook is designed for the aeroelastic QRC workflow in three modes:

1. `USE_HARDWARE = False`: Bloqade Python emulator.
2. `USE_HARDWARE = True`, `USE_MOCK = True`: hardware-like submit/fetch flow with `quera.mock()`.
3. `USE_HARDWARE = True`, `USE_MOCK = False`: real Aquila submission through AWS Braket.

The notebook supports:

- `ENCODING = "local"` or `"global"`
- `QRC_READOUT = "Z"` or `"ZZ"`
- noisy inputs + optional state estimation
- Ridge readout regularization via `QRC_READOUT_ALPHA`
- task/shot budget calculation before submission
- safe confirmation flag before real QPU submission

Required files in the same folder:

- `airfoil_simulator.py`
- `airfoil_qrc_v5.py`
- `airfoil_state_estimation.py`

`airfoil_qrc_v6.py` is not strictly required by this notebook, but it may be kept in the folder.

## Corrections applied (v11)

Fixes so the QRC metric is meaningful (the original run reported MSE ~1e-11, an artifact of testing on the flat initial transient with more embedding features than training samples):

1. **Data** from the *developed* LCO; the subsample stride is chosen automatically so the **test window spans >= 1 full LCO period** (period estimated by FFT), which makes R2 interpretable.
2. **Readout** trained with more samples than the embedding dimension (`MAX_TRAIN_SAMPLES=40`) -> no interpolation.
3. **Honest metrics** after the readout: NRMSE, R2, skill vs persistence **and** vs train-mean.
4. **Diagnostics made kernel-safe**: the sweep/horizon cells (4.1/4.2) now emulate a small fixed window set with a hard cap on emulator runs, so they no longer overload the notebook instance (which caused the Jupyter "Server Connection Error"). They print the planned run count before launching.
5. **Aquila confirmation cells (4.5 submit / 4.6 fetch)** added: a reduced, budget-gated hardware `total_time` scan around the 1 us peak that reuses the existing native submit/fetch path, prints the exact Aquila task/$ cost, and reports whether the emulator advantage survives real decoherence. (A full hardware port of 4.4 would cost ~$1.6k-2.6k; this is the ~$220-360 confirmation.)
6. **`total_time` drill-down (4.4)** added: the decisive significance test — finely varies the reservoir evolution time (8 atoms, Z, 3 seeds, larger test) to see whether the QRC can cross *significantly* above persistence (CI lower bound > 0), with an explicit Aquila go/no-go verdict.
7. **Design sweep (4.3)** added: capped, cached, multi-seed emulator sweep over atoms/readout/shots/total_time to test whether ANY small-atom config makes the QRC robustly beat NG-RC and persistence (with an explicit verdict and the emulator atom-ceiling caveat).
8. **Multi-seed horizon (4.2)**: runs over several noise realizations (`HZ_SEEDS`), pools the test errors, and reports a bootstrap CI plus the per-seed min/max envelope, so the QRC/NG-RC crossover can be shown to hold for every seed (not one lucky draw). Still case-aware:
9. **Case-aware evaluation (set `DYNAMICS`/just change `CASE`)**: the data cell and 4.2 now branch automatically. Periodic cases (I/II/III/V) use the phase-spread test; the chaotic case IV uses a long burn-in, anchors spaced by the measured decorrelation time, a temporal disjoint train/test split, and horizons in multiples of tau_c (decay expected). Both report bootstrap CIs.
10. **Horizon sweep made honest (4.2)**: the test set is now spread across a full LCO period (split by period block, no phase leakage) instead of one narrow arc, so the skill values are not inflated and R2 is interpretable; each horizon gets a 95% bootstrap CI.
11. **Rollout fixed**: the autonomous rollout is bounded to the data envelope (no more inf/overflow crash) and compared against the *true absolute states* (it previously compared absolute-state rollouts to delta targets).
12. **Emulator diagnostics** at the end: a stride x atoms x time_steps x readout sweep and a horizon (H-step-ahead) skill curve. Run on the emulator to pick a config before spending on Aquila.

Config already set: `SMOOTHER_METHOD='quick'`, `MAX_TRAIN_SAMPLES=40`, `MAX_TEST_SAMPLES=15`.

**Hardware budget:** (MAX_TRAIN+MAX_TEST) x QRC_TIME_STEPS = 55 x 2 = 110 Aquila tasks x 30 shots ~ $66.

## Findings summary (state of the investigation)

Primary metric throughout: **skill vs persistence** = 1 - MSE(model)/MSE(predict-no-change), with bootstrap 95% CIs. R2 is reported but is misleading on narrow-phase test sets. "Significant" = CI lower bound > 0. All emulator results are coherent/idealized and **pending hardware confirmation**.

**Case II (periodic LCO).** The 8-atom QRC genuinely predicts (one-step skill ~0.48, significant; beats persistence, train-mean and NG-RC on a single-cycle 70/30 split). **But** with a phase-spread test (train covers the full cycle, the natural setup for a periodic LCO), the classical **NG-RC beats the QRC** (skill ~0.92-0.97 vs ~0.73-0.79). On a single-period limit cycle there is nothing to extrapolate, so a degree-2 polynomial memorizes the cycle. **No quantum advantage on the periodic case.**

**Case IV (chaotic, decorrelation time tau_c ~ 10 tu).**
- *Short horizons (< ~1 tau_c):* NG-RC is better; the QRC is modest.
- *Long horizons (>= ~1.5 tau_c):* the QRC degrades gracefully (skill stays bounded) while NG-RC diverges far below persistence (to ~ -10). Across 3 noise seeds the QRC's worst seed beats NG-RC's best seed from ~1 tau_c onward. **Robust graceful-degradation advantage over NG-RC.**
- *Absolute skill vs persistence at the default `total_time = 4 us`:* multi-seed showed the QRC **below** persistence (negative) at all horizons. The earlier single-seed positive values were a lucky noise draw, not robust.
- *`total_time` drill-down (the decisive test):* skill depends **resonantly** on the reservoir evolution time. At **`total_time ~ 1 us`** the 8-atom QRC **significantly and robustly beats persistence** (skill **0.656**, CI [0.590, 0.713], all 3 seeds positive) and crushes NG-RC (~ -8.9); also positive at 2 us; negative at 1.5, 2.5, 3 us. The "good" times fall at integer multiples of the **Rabi period (2pi/Omega = 1.0 us)** -> evidence that coherent quantum dynamics drive the predictive power. The earlier negative results were an artifact of an over-long (4 us) evolution.

**Status / what can be claimed.**
- Defensible now (emulator): there exists an evolution-time regime (commensurate with the Rabi period) where the small-atom analog QRC predicts chaotic aeroelastic dynamics significantly better than persistence and dramatically better than NG-RC, robustly across noise seeds.
- **Pending:** confirm the 1 us peak survives real decoherence on Aquila (cells 4.5/4.6, reduced scope, ~$220-360). The peak is sharp (0.5 us from +0.66 to -0.91), so hardware calibration matters; scan a few `total_time` around 1 us.
- Caveats: emulator has no T2/atom-loss/detection error; skill is relative to a weak persistence baseline in the long-horizon regime; the local emulator only reaches ~12 atoms, so hardware-scale (tens of atoms) behavior is untested here.


In [3]:

# Optional dependency installation.
# Keep these commented unless your AWS Braket notebook environment is missing packages.
%pip install --upgrade pip
%pip install numpy scipy matplotlib scikit-learn bloqade amazon-braket-sdk boto3 ipykernel

Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached lark-1.3.1-py3-none-any.whl.metadata (1.8 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 147.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 181.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 2

In [5]:

import os
import sys
import json
import warnings
from pathlib import Path
import importlib.util

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge

warnings.filterwarnings(
    "ignore",
    message="pkg_resources is deprecated as an API.*"
)

ROOT = Path.cwd()

def dynamic_import(module_name: str, filepath: str):
    path = ROOT / filepath
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

    spec = importlib.util.spec_from_file_location(module_name, str(path))
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not create import spec for {filepath}")

    mod = importlib.util.module_from_spec(spec)

    # Required for @dataclass and other introspection-based utilities in notebooks.
    sys.modules[module_name] = mod

    spec.loader.exec_module(mod)
    return mod

required_files = [
    "airfoil_simulator.py",
    "airfoil_qrc_v5.py",
    "airfoil_state_estimation.py",
]

for fname in required_files:
    p = ROOT / fname
    print(f"{fname:30s} exists={p.exists()} path={p}")

airfoil_sim = dynamic_import("airfoil_simulator_local", "airfoil_simulator.py")
qrc5 = dynamic_import("airfoil_qrc_v5_local", "airfoil_qrc_v5.py")
state_est = dynamic_import("airfoil_state_est_local", "airfoil_state_estimation.py")

# Reuse validated helper functions from airfoil_qrc_v5.py
QRCConfig = qrc5.QRCConfig
load_or_generate_dataset = qrc5.load_or_generate_dataset
select_state_columns = qrc5.select_state_columns
build_window_dataset = qrc5.build_window_dataset
temporal_split = qrc5.temporal_split
build_ngrc_model = qrc5.build_ngrc_model
autonomous_rollout_classical = qrc5.autonomous_rollout_classical
qrc_scaler_fit_transform = qrc5.qrc_scaler_fit_transform
build_local_task = qrc5.build_local_task
_apply_piecewise_global_detuning = qrc5._apply_piecewise_global_detuning
process_local_report = qrc5.process_local_report
process_results_from_bitstring_list = qrc5.process_results_from_bitstring_list
emulate_qrc_embeddings = qrc5.emulate_qrc_embeddings
autonomous_rollout_qrc = qrc5.autonomous_rollout_qrc
plot_time_history = qrc5.plot_time_history
plot_psd_comparison = qrc5.plot_psd_comparison
plot_phase_portraits = qrc5.plot_phase_portraits
choose_phase_pairs = qrc5.choose_phase_pairs

bloqade, Chain = qrc5.get_bloqade_objects()

print("Imported local project modules successfully.")

airfoil_simulator.py           exists=True path=/home/ec2-user/amazon-braket-examples/examples/analog_hamiltonian_simulation/airfoil_simulator.py
airfoil_qrc_v5.py              exists=True path=/home/ec2-user/amazon-braket-examples/examples/analog_hamiltonian_simulation/airfoil_qrc_v5.py
airfoil_state_estimation.py    exists=True path=/home/ec2-user/amazon-braket-examples/examples/analog_hamiltonian_simulation/airfoil_state_estimation.py
Imported local project modules successfully.


## 1. Configuration

Recommended sequence:

1. `USE_HARDWARE = False`, `USE_MOCK = False` for local emulation.
2. `USE_HARDWARE = True`, `USE_MOCK = True` for hardware-flow mock.
3. `USE_HARDWARE = True`, `USE_MOCK = False`, `CONFIRM_REAL_QPU_SUBMISSION = True` for real Aquila.

For the first real Aquila run, use:

- `CASE = "II"`
- `ENCODING = "local"`
- `QRC_READOUT = "Z"`
- `MAX_TRAIN_SAMPLES = 10`
- `MAX_TEST_SAMPLES = 5`
- `NSHOTS = 30`

In [6]:

# =========================
# User configuration
# =========================

# Execution mode
USE_HARDWARE = True          # False -> local emulator; True -> submit/fetch hardware-style tasks
USE_MOCK = True              # Only used when USE_HARDWARE=True. True -> quera.mock(); False -> real Aquila
# For real QPU + local detuning, use native Braket SDK with experimental_capabilities="ALL".
# This avoids Bloqade wrappers that may not forward experimental capability flags.
USE_NATIVE_BRAKET_FOR_REAL_QPU = True
AQUILA_ARN = "arn:aws:braket:us-east-1::device/qpu/quera/Aquila"
S3_DESTINATION_FOLDER = None  # None -> Braket default bucket/tasks

# Safety switch for real Aquila. Must be True if USE_HARDWARE=True and USE_MOCK=False.
CONFIRM_REAL_QPU_SUBMISSION = False

# QRC configuration
ENCODING = "local"            # "local" or "global"
QRC_READOUT = "Z"             # "Z" or "ZZ"
QRC_TIME_STEPS = 2
QRC_TOTAL_TIME = 4.0
QRC_RABI_FREQUENCY = 6.283
QRC_LATTICE_SPACING = 10.0
QRC_ENCODING_SCALE = 9.0
QRC_PULSE_BIAS = 0.0
QRC_READOUT_ALPHA = 1e-4      # Recommended for noisy hardware embeddings: 1e-4 or 1e-3

# Dataset / aeroelastic setup
CASE = "IV"                   # "II", "III", "IV", etc.
DYNAMICS = "auto"            # "auto" -> chaotic for IV, periodic for I/II/III/V; or force "periodic"/"chaotic"
WINDOW_LENGTH = 2
STATE_INDICES = [0, 1, 2, 3]  # 4 states x window 2 -> 8 atoms
PREDICT_DELTA = True

# Noise / state estimation
USE_NOISY_INPUT = True
NOISE_LEVEL = 0.40
NOISE_KIND = "gaussian"       # "gaussian", "student_t", "none"
STUDENT_DF = 5
USE_STATE_ESTIMATION = True
SMOOTHER_METHOD = "quick"            # use "quick" (fixed_interval is mis-specified once subsampled)
SE_MAX_NFEV = 30

# Train/test sizing
TRAIN_FRACTION = 0.7
MAX_TRAIN_SAMPLES = 40   # must exceed embedding dim (atoms*time_steps)
MAX_TEST_SAMPLES = 15

# Hardware / sampling
NSHOTS = 30
PARALLELIZE_DISTANCE_UM = 15.0

# Reproducibility
SEED = 1234

# Output
OUT_PREFIX = f"aquila_{CASE}_{ENCODING}_{QRC_READOUT}_w{WINDOW_LENGTH}"
TASK_DIR = ROOT / f"{OUT_PREFIX}_tasks"
RESULT_DIR = ROOT / f"{OUT_PREFIX}_results"

assert ENCODING in ("local", "global")
assert QRC_READOUT in ("Z", "ZZ")

print("Configuration loaded.")
print("OUT_PREFIX:", OUT_PREFIX)

Configuration loaded.
OUT_PREFIX: aquila_IV_local_Z_w2


In [7]:

# Optional AWS credentials.
# Usually not needed inside an AWS Braket notebook instance if IAM role/profile is configured.
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."
# os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

In [8]:
# =========================
# Task/shot budget
# =========================

def estimate_task_budget():
    n_points = int(MAX_TRAIN_SAMPLES + MAX_TEST_SAMPLES)

    # In Bloqade local mock/emulator, one task-like object can contain all probe times.
    # In native Braket real-QPU mode with local detuning, this notebook submits one
    # AHS task per data point per probe time. This is safer and makes explicit the
    # intermediate-time readout used for QRC embeddings.
    native_real_local = bool(
        USE_HARDWARE and (not USE_MOCK) and ENCODING == "local" and USE_NATIVE_BRAKET_FOR_REAL_QPU
    )

    if ENCODING == "local" and not native_real_local:
        n_tasks = n_points
        task_multiplier_reason = "Bloqade local batch/probe-time task object"
    else:
        n_tasks = n_points * int(QRC_TIME_STEPS)
        task_multiplier_reason = "one real AHS task per data point per probe time"

    requested_shots = n_tasks * int(NSHOTS)

    budget = {
        "case": CASE,
        "encoding": ENCODING,
        "train_samples": int(MAX_TRAIN_SAMPLES),
        "test_samples": int(MAX_TEST_SAMPLES),
        "n_data_points": n_points,
        "qrc_time_steps": int(QRC_TIME_STEPS),
        "tasks_to_submit": int(n_tasks),
        "shots_per_task": int(NSHOTS),
        "requested_shots_total": int(requested_shots),
        "task_multiplier_reason": task_multiplier_reason,
        "use_hardware": bool(USE_HARDWARE),
        "use_mock": bool(USE_MOCK),
        "native_braket_real_qpu": bool(native_real_local),
        "real_qpu": bool(USE_HARDWARE and not USE_MOCK),
    }
    return budget

budget = estimate_task_budget()
print(json.dumps(budget, indent=2))

{
  "case": "IV",
  "encoding": "local",
  "train_samples": 40,
  "test_samples": 15,
  "n_data_points": 55,
  "qrc_time_steps": 2,
  "tasks_to_submit": 55,
  "shots_per_task": 30,
  "requested_shots_total": 1650,
  "task_multiplier_reason": "Bloqade local batch/probe-time task object",
  "use_hardware": true,
  "use_mock": true,
  "native_braket_real_qpu": false,
  "real_qpu": false
}


In [9]:
# =========================
# Build dataset: DEVELOPED-LCO (periodic cases) OR ATTRACTOR segment (chaotic case IV)
# =========================
# Periodic cases (I/II/III/V): skip a burn-in into the developed LCO and pick the
# subsample stride so the TEST window spans >= 1 LCO period (period via FFT).
# Chaotic case (IV): there is no period. Use a long burn-in onto the attractor and
# a stride tied to the DECORRELATION TIME (autocorrelation 1/e), so one-step targets
# are meaningful; the temporal train/test split is already a genuine disjoint test.

SEGMENT_BURN_IN_TU    = 300.0    # periodic cases
TEST_PERIODS_COVERAGE = 1.0      # periodic: test must span >= this many periods
LCO_PERIOD_TU         = None     # periodic: None -> FFT estimate
SEGMENT_STRIDE        = "auto"   # periodic: "auto" -> from period; or an int

CHAOS_BURN_IN_TU      = 500.0    # chaotic: longer, to settle onto the attractor
CHAOS_STRIDE_FRAC     = 0.2      # chaotic: eff_dt = this * decorrelation_time

# resolve dynamics regime
_chaotic = (DYNAMICS == "chaotic") or (DYNAMICS == "auto" and str(CASE).upper() == "IV")

data = load_or_generate_dataset(
    input_npz=None, case=CASE,
    noise_level=NOISE_LEVEL if USE_NOISY_INPUT else 0.0,
    noise_kind=NOISE_KIND if USE_NOISY_INPUT else "none",
    student_df=STUDENT_DF, seed=SEED,
)
X_clean_all = np.asarray(data["X_sampled"], dtype=float)
X_obs_all   = np.asarray(data["Y_sampled"], dtype=float)
tau_all     = np.asarray(data["tau_sampled"], dtype=float)
metadata = data.get("metadata", {})
dt_sample_raw = float(metadata.get("sampling_interval", np.median(np.diff(tau_all))))

def _char_time_tu(x, dt, max_lag_tu=150.0):
    """Decorrelation time (autocorrelation 1/e crossing), in time units."""
    x = np.asarray(x, float) - np.mean(x)
    n = len(x); ml = min(n - 1, int(round(max_lag_tu / dt)))
    ac = np.correlate(x, x, "full")[n - 1: n - 1 + ml]
    if ac[0] <= 0:
        return None
    ac = ac / ac[0]
    below = np.where(ac < 1.0 / np.e)[0]
    return float(below[0] * dt) if len(below) else None

def _period_tu(x, dt):
    x = np.asarray(x, float) - np.mean(x)
    if len(x) < 32: return None
    F = np.abs(np.fft.rfft(x * np.hanning(len(x)))); f = np.fft.rfftfreq(len(x), d=dt); F[0] = 0.0
    k = int(np.argmax(F)); return (1.0 / f[k]) if f[k] > 0 else None

if _chaotic:
    burn_tu = CHAOS_BURN_IN_TU
    i0 = int(round(burn_tu / dt_sample_raw))
    tau_c = _char_time_tu(X_clean_all[i0:i0 + min(int(round(600 / dt_sample_raw)), len(X_clean_all) - i0), 0],
                          dt_sample_raw) or 10.0
    eff_dt_target = CHAOS_STRIDE_FRAC * tau_c
    SEGMENT_STRIDE_EFF = max(1, int(round(eff_dt_target / dt_sample_raw)))
    char_tu = tau_c
    regime = f"chaotic (decorrelation time ~ {tau_c:.1f} tu)"
else:
    burn_tu = SEGMENT_BURN_IN_TU
    i0 = int(round(burn_tu / dt_sample_raw))
    if LCO_PERIOD_TU is None:
        _probe = X_clean_all[i0: i0 + min(int(round(300 / dt_sample_raw)), len(X_clean_all) - i0), 0]
        char_tu = _period_tu(_probe, dt_sample_raw)
    else:
        char_tu = float(LCO_PERIOD_TU)
    if isinstance(SEGMENT_STRIDE, str) and SEGMENT_STRIDE.lower() == "auto":
        if char_tu is None:
            SEGMENT_STRIDE_EFF = 30
        else:
            SEGMENT_STRIDE_EFF = max(1, int(round(
                (TEST_PERIODS_COVERAGE * char_tu / max(1, MAX_TEST_SAMPLES)) / dt_sample_raw)))
    else:
        SEGMENT_STRIDE_EFF = int(SEGMENT_STRIDE)
    regime = f"periodic (LCO period ~ {char_tu:.1f} tu)" if char_tu else "periodic (period unknown)"

n_windows_needed = int(np.ceil(max(MAX_TRAIN_SAMPLES / max(1e-9, TRAIN_FRACTION),
                                   MAX_TEST_SAMPLES / max(1e-9, 1.0 - TRAIN_FRACTION))))
n_sub_needed = WINDOW_LENGTH + n_windows_needed + 10
raw_needed = n_sub_needed * SEGMENT_STRIDE_EFF
seg_idx = np.arange(i0, min(i0 + raw_needed, len(X_clean_all)), SEGMENT_STRIDE_EFF, dtype=int)
if len(seg_idx) < (WINDOW_LENGTH + 5):
    raise ValueError("Segment too short; reduce burn-in / stride.")

X_clean_full = X_clean_all[seg_idx].copy()
X_obs_full   = X_obs_all[seg_idx].copy()
tau_sampled  = tau_all[seg_idx].copy()
dt_sample = dt_sample_raw * SEGMENT_STRIDE_EFF

_sel = X_clean_full[:, STATE_INDICES]; _d = np.diff(_sel, axis=0)
print(f"Regime: {regime}")
print(f"  burn-in {burn_tu} tu, stride {SEGMENT_STRIDE_EFF}, eff_dt {dt_sample:.3f}, points {len(seg_idx)}")
print(f"  tau span: [{tau_sampled[0]:.1f}, {tau_sampled[-1]:.1f}]")
if not _chaotic and char_tu:
    _ts = MAX_TEST_SAMPLES * dt_sample
    print(f"  test window spans {_ts:.1f} tu = {_ts/char_tu:.2f} periods")
print(f"  predict-zero baseline MSE: {np.mean(_d**2):.3e}   target std: {np.std(_d, axis=0)}")
if SMOOTHER_METHOD == "fixed_interval":
    print("  WARNING: set SMOOTHER_METHOD='quick'.")


Regime: chaotic (decorrelation time ~ 10.3 tu)
  burn-in 500.0 tu, stride 41, eff_dt 2.050, points 70
  tau span: [500.0, 641.5]
  predict-zero baseline MSE: 1.371e-03   target std: [0.03469674 0.00647449 0.06277447 0.00579479]


In [10]:

# =========================
# Optional state estimation on cropped prefix only
# =========================

X_for_learning_full = X_obs_full.copy()
state_estimation_info = None

if USE_NOISY_INPUT and USE_STATE_ESTIMATION:
    X_est_full, state_estimation_info = state_est.estimate_states_from_observations(
        Y_obs=X_obs_full,
        params=data["params"],
        method=SMOOTHER_METHOD,
        dt_sample=dt_sample,
        max_nfev=SE_MAX_NFEV,
        X_true=X_clean_full,
    )
    X_for_learning_full = np.asarray(X_est_full, dtype=float)

print(
    "Using input source:",
    "estimated states" if (USE_NOISY_INPUT and USE_STATE_ESTIMATION)
    else ("noisy observations" if USE_NOISY_INPUT else "clean states")
)

if state_estimation_info is not None:
    print(json.dumps(state_estimation_info, indent=2))

Using input source: estimated states
{
  "method": "savgol",
  "window_length": 11,
  "polyorder": 3,
  "mse_vs_truth": 0.0022726452183178428,
  "rmse_vs_truth": 0.047672268860605355
}


In [11]:

# =========================
# Select states and build windows
# =========================

X_clean_sel = select_state_columns(X_clean_full, STATE_INDICES)
X_in_sel = select_state_columns(X_for_learning_full, STATE_INDICES)

windows, targets, current = build_window_dataset(
    X_input=X_in_sel,
    X_target=X_clean_sel,
    window_length=WINDOW_LENGTH,
    predict_delta=PREDICT_DELTA,
)

split = temporal_split(
    windows=windows,
    targets=targets,
    current=current,
    train_fraction=TRAIN_FRACTION,
)

X_train = split["X_train"][:MAX_TRAIN_SAMPLES]
Y_train = split["Y_train"][:MAX_TRAIN_SAMPLES]
C_train = split["C_train"][:MAX_TRAIN_SAMPLES]

X_test = split["X_test"][:MAX_TEST_SAMPLES]
Y_test = split["Y_test"][:MAX_TEST_SAMPLES]
C_test = split["C_test"][:MAX_TEST_SAMPLES]

state_dim = len(STATE_INDICES)
atom_number = X_train.shape[1]

print("Train windows:", X_train.shape, "Train targets:", Y_train.shape)
print("Test windows :", X_test.shape, "Test targets :", Y_test.shape)
print("State dim    :", state_dim, "Atom number:", atom_number)

assert len(X_train) > 0 and len(X_test) > 0
assert atom_number == WINDOW_LENGTH * state_dim

Train windows: (40, 8) Train targets: (40, 4)
Test windows : (15, 8) Test targets : (15, 4)
State dim    : 4 Atom number: 8


In [12]:

# =========================
# Baseline NG-RC
# =========================

NGRC_ALPHA = 1e-4

ngrc_model = build_ngrc_model(degree=2)
ngrc_model.fit(X_train, Y_train)

Y_pred_ngrc = ngrc_model.predict(X_test)
mse_one_step_ngrc_total = float(np.mean((Y_pred_ngrc - Y_test) ** 2))
mse_one_step_ngrc_alpha = float(np.mean((Y_pred_ngrc[:, 0] - Y_test[:, 0]) ** 2))

print("NG-RC alpha:", NGRC_ALPHA)
print("Baseline NG-RC one-step total MSE:", mse_one_step_ngrc_total)
print("Baseline NG-RC one-step alpha MSE:", mse_one_step_ngrc_alpha)

NG-RC alpha: 0.0001
Baseline NG-RC one-step total MSE: 0.0001006639209954429
Baseline NG-RC one-step alpha MSE: 0.0001771490978605541


In [13]:

# =========================
# QRC config + scaling
# =========================

qrc = QRCConfig(
    atom_number=atom_number,
    encoding=ENCODING,
    lattice_spacing=QRC_LATTICE_SPACING,
    encoding_scale=QRC_ENCODING_SCALE,
    rabi_frequency=QRC_RABI_FREQUENCY,
    total_time=QRC_TOTAL_TIME,
    time_steps=QRC_TIME_STEPS,
    readouts=QRC_READOUT,
    pulse_bias=QRC_PULSE_BIAS,
)

# airfoil_qrc_v5.py returns: X_train_scaled, X_test_scaled, scaler
X_train_scaled, X_test_scaled, scaler = qrc_scaler_fit_transform(X_train, X_test)

X_train_scaled = np.asarray(X_train_scaled, dtype=float)
X_test_scaled = np.asarray(X_test_scaled, dtype=float)

print("QRC atom number:", qrc.atom_number)
print("Encoding:", qrc.encoding)
print("Readout:", qrc.readouts)
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape :", X_test_scaled.shape)
print("Scaler type:", type(scaler))

QRC atom number: 8
Encoding: local
Readout: Z
X_train_scaled shape: (40, 8)
X_test_scaled shape : (15, 8)
Scaler type: <class 'sklearn.preprocessing._data.MinMaxScaler'>


In [14]:

# =========================
# Emulator helper
# =========================

def emulate_embeddings_dispatch(qrc, X_scaled, nshots):
    if qrc.encoding == "local":
        return emulate_qrc_embeddings(qrc, X_scaled, nshots=nshots)

    # Global encoding: emulate one datapoint by collecting one report per probe time.
    embeddings = []
    for row in X_scaled:
        reports = qrc5.build_global_task_reports(qrc, row, nshots=nshots)
        bits_list = [rep.bitstrings() for rep in reports]
        emb = process_results_from_bitstring_list(qrc, bits_list)
        embeddings.append(emb)
    return np.asarray(embeddings, dtype=float)


### Hardware-local detuning fix

For real Aquila execution with `ENCODING="local"`, this notebook uses the native Braket SDK path and submits tasks with `experimental_capabilities="ALL"`. Local detuning on Aquila is an experimental capability and must be explicitly enabled on each real QPU task. The hardware path also uses a ramp-up/hold/ramp-down local detuning waveform because the local detuning time series must start and end at zero.

In native real-QPU mode, the notebook submits one AHS task per data point per QRC probe time. Therefore, for `10` train samples, `5` test samples, and `QRC_TIME_STEPS=4`, the budget is `60` real QPU tasks rather than `15` Bloqade mock task objects.


In [15]:
# =========================
# Hardware helpers
# =========================

TASK_DIR.mkdir(exist_ok=True, parents=True)
RESULT_DIR.mkdir(exist_ok=True, parents=True)

# Native Braket imports are only required for real Aquila execution.
def _import_native_braket():
    from braket.aws import AwsDevice, AwsQuantumTask
    from braket.ahs.atom_arrangement import AtomArrangement
    from braket.ahs.driving_field import DrivingField
    from braket.ahs.local_detuning import LocalDetuning
    from braket.ahs.field import Field
    from braket.ahs.pattern import Pattern
    from braket.ahs.analog_hamiltonian_simulation import AnalogHamiltonianSimulation
    from braket.timings.time_series import TimeSeries
    return {
        "AwsDevice": AwsDevice,
        "AwsQuantumTask": AwsQuantumTask,
        "AtomArrangement": AtomArrangement,
        "DrivingField": DrivingField,
        "LocalDetuning": LocalDetuning,
        "Field": Field,
        "Pattern": Pattern,
        "AnalogHamiltonianSimulation": AnalogHamiltonianSimulation,
        "TimeSeries": TimeSeries,
    }


def build_global_program_for_probe(qrc, x_scaled, probe_idx):
    geom = Chain(qrc.atom_number, lattice_spacing=qrc.lattice_spacing)

    seq = np.asarray(x_scaled, dtype=float)
    if len(seq) < 2:
        seq = np.concatenate([seq, seq])

    segment_duration = qrc.total_time / len(seq)
    durations_full = [segment_duration] * (len(seq) - 1)
    values_full = list(qrc.pulse_bias + qrc.encoding_scale * (2.0 * seq - 1.0))

    prefix_len = max(2, int(np.ceil(len(seq) * probe_idx / qrc.time_steps)))
    durations = durations_full[: prefix_len - 1]
    values = values_full[:prefix_len]

    base = geom.rydberg.rabi.amplitude.uniform.constant(
        duration=float(np.sum(durations)),
        value=qrc.rabi_frequency,
    )
    program = _apply_piecewise_global_detuning(base.detuning.uniform, durations, values)
    return program


def _run_async_program(program, nshots, name, use_mock):
    """
    Run a Bloqade program either on QuEra mock or on real Aquila.

    IMPORTANT:
    For real Aquila + local detuning, Bloqade expects use_experimental=True.
    Do NOT pass experimental_capabilities="ALL" here; that is for native Braket SDK.
    """
    if use_mock:
        return program.quera.mock().run_async(
            shots=nshots,
            name=name,
        )

    # Real Aquila through Bloqade wrapper.
    # Local detuning requires experimental capabilities.
    try:
        return program.braket.aquila().run_async(
            shots=nshots,
            name=name,
            use_experimental=True,
        )
    except TypeError as exc:
        raise RuntimeError(
            "Bloqade did not accept use_experimental=True. "
            "Use the native Braket route instead by setting "
            "USE_NATIVE_BRAKET_FOR_REAL_QPU=True."
        ) from exc


def _build_native_local_detuning_ahs(qrc, x_scaled, probe_idx):
    """Build a hardware-valid native Braket AHS program with local detuning.

    The QRC emulator used a constant local detuning. Aquila's local detuning
    experimental capability requires a time series that starts and ends at zero.
    Therefore, the hardware version uses a short ramp-up/hold/ramp-down profile,
    while keeping the spatial pattern equal to the scaled QRC input vector.
    """
    b = _import_native_braket()
    AtomArrangement = b["AtomArrangement"]
    DrivingField = b["DrivingField"]
    LocalDetuning = b["LocalDetuning"]
    Field = b["Field"]
    Pattern = b["Pattern"]
    AnalogHamiltonianSimulation = b["AnalogHamiltonianSimulation"]
    TimeSeries = b["TimeSeries"]

    x_scaled = np.asarray(x_scaled, dtype=float)
    pattern_values = np.clip(x_scaled, 0.0, 1.0).tolist()

    # Bloqade values are in rad / microsecond and microseconds.
    # Native Braket AHS requires SI units: rad / second and seconds.
    run_time_us = float(qrc.total_time) * float(probe_idx) / float(qrc.time_steps)
    time_max = run_time_us * 1e-6
    time_ramp = min(0.20e-6, time_max / 4.0)
    if time_max <= 2.0 * time_ramp:
        time_ramp = time_max / 4.0

    omega_max = float(qrc.rabi_frequency) * 1e6
    delta_global = float(qrc.encoding_scale / 2.0) * 1e6
    # Keep the local detuning sign consistent with the Bloqade notebook:
    # .scale(x).constant(value=-encoding_scale)
    delta_local = -abs(float(qrc.encoding_scale) * 1e6)

    register = AtomArrangement()
    spacing_m = float(qrc.lattice_spacing) * 1e-6
    for k in range(qrc.atom_number):
        register.add((k * spacing_m, 0.0))

    omega = TimeSeries.from_lists(
        times=[0.0, time_ramp, max(time_ramp, time_max - time_ramp), time_max],
        values=[0.0, omega_max, omega_max, 0.0],
    )
    phase = TimeSeries.from_lists(times=[0.0, time_max], values=[0.0, 0.0])
    detuning = TimeSeries.from_lists(times=[0.0, time_max], values=[delta_global, delta_global])

    # Local detuning must start and end at zero. The spatial pattern remains static.
    local_ts = TimeSeries.from_lists(
        times=[0.0, time_ramp, max(time_ramp, time_max - time_ramp), time_max],
        values=[0.0, delta_local, delta_local, 0.0],
    )

    drive = DrivingField(amplitude=omega, phase=phase, detuning=detuning)
    local_shift = LocalDetuning(
        magnitude=Field(
            time_series=local_ts,
            pattern=Pattern(pattern_values),
        )
    )

    ahs_program = AnalogHamiltonianSimulation(
        register=register,
        hamiltonian=drive + local_shift,
    )
    return ahs_program


def _submit_native_local_detuning_task(qrc, x_scaled, probe_idx, nshots, name):
    b = _import_native_braket()
    AwsDevice = b["AwsDevice"]
    device = AwsDevice(AQUILA_ARN)
    ahs_program = _build_native_local_detuning_ahs(qrc, x_scaled, probe_idx)
    ahs_program = ahs_program.discretize(device)
    task = device.run(
        ahs_program,
        s3_destination_folder=S3_DESTINATION_FOLDER,
        shots=int(nshots),
        experimental_capabilities="ALL",
    )
    return task


def _save_native_task_metadata(task, path, metadata=None):
    payload = {
        "type": "native_braket_ahs_task",
        "arn": task.id,
        "metadata": metadata or {},
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


def _load_native_task(path):
    b = _import_native_braket()
    AwsQuantumTask = b["AwsQuantumTask"]
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)
    return AwsQuantumTask(payload["arn"]), payload


def _ahs_result_to_rydberg_bits(result, natoms):
    """Convert Braket AHS measurements to Rydberg-occupancy bitstrings.

    Braket AHS post_sequence convention: 0 = Rydberg or empty, 1 = ground.
    We keep only successfully prepared atoms. For a fully prepared shot,
    rydberg_bit = 1 - post_sequence.
    """
    rows = []
    for shot in result.measurements:
        status = str(getattr(shot, "status", ""))
        # status may be an enum; compare robustly.
        if "Success" not in status:
            continue
        pre = np.asarray(shot.pre_sequence, dtype=int)
        post = np.asarray(shot.post_sequence, dtype=int)
        if pre.shape[0] != natoms or post.shape[0] != natoms:
            continue
        if not np.all(pre == 1):
            # skip imperfect initialization for this first verification notebook
            continue
        rows.append(1 - post)
    if not rows:
        return np.zeros((1, natoms), dtype=int)
    return np.asarray(rows, dtype=int)


def submit_local_tasks_hardware(qrc, X_scaled, prefix, nshots, use_mock=False):
    task_paths = []

    for idx, row in enumerate(X_scaled):

        if (not use_mock) and USE_NATIVE_BRAKET_FOR_REAL_QPU:
            # Native Braket path:
            # one real QPU task per data point per probe time.
            for probe_idx in range(1, qrc.time_steps + 1):
                task = _submit_native_local_detuning_task(
                    qrc,
                    row,
                    probe_idx,
                    nshots=nshots,
                    name=f"{prefix}_{idx}_t{probe_idx}",
                )

                path = TASK_DIR / f"{prefix}_{idx}_t{probe_idx}.json"

                _save_native_task_metadata(
                    task,
                    path,
                    metadata={
                        "prefix": prefix,
                        "data_index": idx,
                        "probe_idx": probe_idx,
                    },
                )

                task_paths.append(path)
                print(f"Submitted native local {prefix}_{idx}_t{probe_idx} -> {task.id}")

        else:
            # Bloqade route:
            # - In mock mode, keep parallelize(...) because it is useful and does not hit Aquila geometry limits.
            # - In real Aquila mode, do NOT use parallelize(...) in the first hardware run.
            #   It made the atom arrangement too tall for Aquila.
            if use_mock:
                program = build_local_task(qrc, row).parallelize(PARALLELIZE_DISTANCE_UM)
            else:
                program = build_local_task(qrc, row)

            task = _run_async_program(
                program,
                nshots=nshots,
                name=f"{prefix}_{idx}",
                use_mock=use_mock,
            )

            path = TASK_DIR / f"{prefix}_{idx}.json"
            bloqade.analog.save(task, str(path))
            task_paths.append(path)

            print(f"Submitted {prefix}_{idx} -> {path.name}")

    return task_paths


def fetch_local_embeddings_hardware(qrc, prefix, count):
    embeddings = []
    native_real_local = bool(
        USE_HARDWARE and (not USE_MOCK) and USE_NATIVE_BRAKET_FOR_REAL_QPU
    )
    for idx in range(count):
        if native_real_local:
            bitstrings_by_time = []
            for probe_idx in range(1, qrc.time_steps + 1):
                task_path = TASK_DIR / f"{prefix}_{idx}_t{probe_idx}.json"
                task, payload = _load_native_task(task_path)
                result = task.result()
                out_path = RESULT_DIR / f"{prefix}_{idx}_t{probe_idx}_result.json"
                with open(out_path, "w", encoding="utf-8") as f:
                    json.dump({"arn": payload["arn"], "metadata": payload.get("metadata", {})}, f, indent=2)
                bits = _ahs_result_to_rydberg_bits(result, qrc.atom_number)
                bitstrings_by_time.append(bits)
                print(f"Fetched native {prefix}_{idx}_t{probe_idx} -> bits {bits.shape}")
            emb = process_results_from_bitstring_list(qrc, bitstrings_by_time)
        else:
            task_path = TASK_DIR / f"{prefix}_{idx}.json"
            task = bloqade.analog.load(str(task_path))
            result = task.fetch()
            out_path = RESULT_DIR / f"{prefix}_{idx}.json"
            bloqade.analog.save(result, str(out_path))
            report = bloqade.analog.load(str(out_path)).report()
            emb = process_local_report(qrc, report)
        embeddings.append(emb)
        print(f"Fetched {prefix}_{idx} -> embedding dim {len(emb)}")
    return np.asarray(embeddings, dtype=float)


def submit_global_tasks_hardware(qrc, X_scaled, prefix, nshots, use_mock=False):
    if (not use_mock) and USE_NATIVE_BRAKET_FOR_REAL_QPU:
        raise NotImplementedError(
            "This patched notebook implements native real-QPU execution for local detuning only. "
            "Use ENCODING='local' for Aquila real runs, or USE_MOCK=True for global-flow testing."
        )
    task_paths = []
    for idx, row in enumerate(X_scaled):
        for probe_idx in range(1, qrc.time_steps + 1):
            program = build_global_program_for_probe(qrc, row, probe_idx).parallelize(PARALLELIZE_DISTANCE_UM)
            task = _run_async_program(program, nshots=nshots, name=f"{prefix}_{idx}_t{probe_idx}", use_mock=use_mock)
            path = TASK_DIR / f"{prefix}_{idx}_t{probe_idx}.json"
            bloqade.analog.save(task, str(path))
            task_paths.append(path)
            print(f"Submitted {prefix}_{idx}_t{probe_idx} -> {path.name}")
    return task_paths


def fetch_global_embeddings_hardware(qrc, prefix, count):
    embeddings = []
    for idx in range(count):
        bitstrings_by_time = []
        for probe_idx in range(1, qrc.time_steps + 1):
            task_path = TASK_DIR / f"{prefix}_{idx}_t{probe_idx}.json"
            task = bloqade.analog.load(str(task_path))
            result = task.fetch()
            out_path = RESULT_DIR / f"{prefix}_{idx}_t{probe_idx}.json"
            bloqade.analog.save(result, str(out_path))
            report = bloqade.analog.load(str(out_path)).report()
            bitstrings_by_time.append(report.bitstrings())
        emb = process_results_from_bitstring_list(qrc, bitstrings_by_time)
        embeddings.append(emb)
        print(f"Fetched {prefix}_{idx} -> embedding dim {len(emb)}")
    return np.asarray(embeddings, dtype=float)

## 2. Obtain QRC embeddings

This cell either:

- runs the emulator, or
- submits hardware/mock tasks.

If `USE_HARDWARE=True`, this cell only submits tasks. Run the next cell after the tasks have completed.

In [16]:

# =========================
# Obtain QRC embeddings: emulator or task submission
# =========================

if USE_HARDWARE and (not USE_MOCK) and (not CONFIRM_REAL_QPU_SUBMISSION):
    raise RuntimeError(
        "Real Aquila submission is disabled by safety flag. "
        "Set CONFIRM_REAL_QPU_SUBMISSION=True after checking the task/shot budget."
    )

if not USE_HARDWARE:
    print("Running Bloqade Python emulator.")
    E_train = emulate_embeddings_dispatch(qrc, X_train_scaled, nshots=NSHOTS)
    E_test = emulate_embeddings_dispatch(qrc, X_test_scaled, nshots=NSHOTS)
    print("E_train shape:", E_train.shape)
    print("E_test  shape:", E_test.shape)
else:
    print("Submitting hardware-style tasks.")
    print(json.dumps(estimate_task_budget(), indent=2))
    if qrc.encoding == "local":
        _ = submit_local_tasks_hardware(qrc, X_train_scaled, prefix="qrc_train", nshots=NSHOTS, use_mock=USE_MOCK)
        _ = submit_local_tasks_hardware(qrc, X_test_scaled, prefix="qrc_test", nshots=NSHOTS, use_mock=USE_MOCK)
        print("Submitted local tasks. Wait for completion, then run the fetch cell.")
    else:
        _ = submit_global_tasks_hardware(qrc, X_train_scaled, prefix="qrc_train", nshots=NSHOTS, use_mock=USE_MOCK)
        _ = submit_global_tasks_hardware(qrc, X_test_scaled, prefix="qrc_test", nshots=NSHOTS, use_mock=USE_MOCK)
        print("Submitted global tasks. Wait for completion, then run the fetch cell.")

Submitting hardware-style tasks.
{
  "case": "IV",
  "encoding": "local",
  "train_samples": 40,
  "test_samples": 15,
  "n_data_points": 55,
  "qrc_time_steps": 2,
  "tasks_to_submit": 55,
  "shots_per_task": 30,
  "requested_shots_total": 1650,
  "task_multiplier_reason": "Bloqade local batch/probe-time task object",
  "use_hardware": true,
  "use_mock": true,
  "native_braket_real_qpu": false,
  "real_qpu": false
}
Submitted qrc_train_0 -> qrc_train_0.json
Submitted qrc_train_1 -> qrc_train_1.json
Submitted qrc_train_2 -> qrc_train_2.json
Submitted qrc_train_3 -> qrc_train_3.json
Submitted qrc_train_4 -> qrc_train_4.json
Submitted qrc_train_5 -> qrc_train_5.json
Submitted qrc_train_6 -> qrc_train_6.json
Submitted qrc_train_7 -> qrc_train_7.json
Submitted qrc_train_8 -> qrc_train_8.json
Submitted qrc_train_9 -> qrc_train_9.json
Submitted qrc_train_10 -> qrc_train_10.json
Submitted qrc_train_11 -> qrc_train_11.json
Submitted qrc_train_12 -> qrc_train_12.json
Submitted qrc_train_13 -> 

In [17]:

# =========================
# Fetch hardware/mock embeddings
# =========================

# Run this cell only after hardware/mock tasks have completed when USE_HARDWARE=True.
# If USE_HARDWARE=False, E_train/E_test already exist from the emulator cell.

if USE_HARDWARE:
    if qrc.encoding == "local":
        E_train = fetch_local_embeddings_hardware(qrc, prefix="qrc_train", count=len(X_train_scaled))
        E_test = fetch_local_embeddings_hardware(qrc, prefix="qrc_test", count=len(X_test_scaled))
    else:
        E_train = fetch_global_embeddings_hardware(qrc, prefix="qrc_train", count=len(X_train_scaled))
        E_test = fetch_global_embeddings_hardware(qrc, prefix="qrc_test", count=len(X_test_scaled))

print("E_train shape:", E_train.shape)
print("E_test  shape:", E_test.shape)
assert E_train.shape[0] == len(Y_train)
assert E_test.shape[0] == len(Y_test)

Fetched qrc_train_0 -> embedding dim 16
Fetched qrc_train_1 -> embedding dim 16
Fetched qrc_train_2 -> embedding dim 16
Fetched qrc_train_3 -> embedding dim 16
Fetched qrc_train_4 -> embedding dim 16
Fetched qrc_train_5 -> embedding dim 16
Fetched qrc_train_6 -> embedding dim 16
Fetched qrc_train_7 -> embedding dim 16
Fetched qrc_train_8 -> embedding dim 16
Fetched qrc_train_9 -> embedding dim 16
Fetched qrc_train_10 -> embedding dim 16
Fetched qrc_train_11 -> embedding dim 16
Fetched qrc_train_12 -> embedding dim 16
Fetched qrc_train_13 -> embedding dim 16
Fetched qrc_train_14 -> embedding dim 16
Fetched qrc_train_15 -> embedding dim 16
Fetched qrc_train_16 -> embedding dim 16
Fetched qrc_train_17 -> embedding dim 16
Fetched qrc_train_18 -> embedding dim 16
Fetched qrc_train_19 -> embedding dim 16
Fetched qrc_train_20 -> embedding dim 16
Fetched qrc_train_21 -> embedding dim 16
Fetched qrc_train_22 -> embedding dim 16
Fetched qrc_train_23 -> embedding dim 16
Fetched qrc_train_24 -> em

In [18]:

# =========================
# Train QRC readout
# =========================

qrc_readout = Ridge(alpha=QRC_READOUT_ALPHA, fit_intercept=True)
qrc_readout.fit(E_train, Y_train)

Y_pred_qrc = qrc_readout.predict(E_test)
mse_one_step_qrc_total = float(np.mean((Y_pred_qrc - Y_test) ** 2))
mse_one_step_qrc_alpha = float(np.mean((Y_pred_qrc[:, 0] - Y_test[:, 0]) ** 2))

print("QRC readout alpha:", QRC_READOUT_ALPHA)
print("QRC one-step total MSE:", mse_one_step_qrc_total)
print("QRC one-step alpha MSE:", mse_one_step_qrc_alpha)

QRC readout alpha: 0.0001
QRC one-step total MSE: 0.0022556277114967856
QRC one-step alpha MSE: 0.0011670941830736307


### Honest metrics (NRMSE, R2, skill vs persistence & mean)

In [19]:
# =========================
# Honest one-step metrics: NRMSE, R2, skill vs persistence AND vs train-mean
# =========================
import pandas as pd
from sklearn.metrics import r2_score

Y_pred_qrc  = qrc_readout.predict(E_test)
Y_pred_ngrc = ngrc_model.predict(X_test)

# Two trivial baselines on the SAME targets:
#   persistence -> predict_delta=True => delta=0 (x_{n+1}=x_n); else last window frame.
#   train-mean  -> predict the average training target.
if PREDICT_DELTA:
    Y_pred_persist = np.zeros_like(Y_test)
else:
    Y_pred_persist = X_test.reshape(len(X_test), WINDOW_LENGTH, len(STATE_INDICES))[:, -1, :]
Y_pred_mean = np.broadcast_to(Y_train.mean(axis=0), Y_test.shape)

mse_persist = float(np.mean((Y_pred_persist - Y_test) ** 2))
mse_mean    = float(np.mean((Y_pred_mean - Y_test) ** 2))

if len(E_train) <= E_train.shape[1]:
    print(f"WARNING: train samples ({len(E_train)}) <= embedding dim ({E_train.shape[1]}). "
          f"Readout is underdetermined and will interpolate; metrics may be optimistic.")

state_names = [["alpha", "alpha_dot", "xi", "xi_dot", "w1", "w2"][i] for i in STATE_INDICES]

def _metrics(name, Yhat):
    mse = float(np.mean((Yhat - Y_test) ** 2))
    rmse = float(np.sqrt(mse))
    denom = float(np.std(Y_test))
    nrmse = rmse / denom if denom > 1e-12 else np.nan
    r2 = float(r2_score(Y_test, Yhat, multioutput="variance_weighted"))
    skill_p = 1.0 - mse / mse_persist if mse_persist > 1e-30 else np.nan
    skill_m = 1.0 - mse / mse_mean if mse_mean > 1e-30 else np.nan
    return {"model": name, "MSE": mse, "RMSE": rmse, "NRMSE": nrmse, "R2": r2,
            "skill_vs_persistence": skill_p, "skill_vs_mean": skill_m}

rows = [
    _metrics("QRC (Aquila)", Y_pred_qrc),
    _metrics("NG-RC", Y_pred_ngrc),
    _metrics("persistence (delta=0)", Y_pred_persist),
    _metrics("train-mean", Y_pred_mean),
]
df_metrics = pd.DataFrame(rows).set_index("model")
pd.set_option("display.float_format", lambda v: f"{v:.4e}")
print("=== One-step test metrics (lower MSE/NRMSE, higher R2/skill is better) ===")
print(df_metrics)

print("\nPer-state R2 (QRC vs NG-RC vs persistence):")
per_state = pd.DataFrame({
    "QRC":         r2_score(Y_test, Y_pred_qrc,  multioutput="raw_values"),
    "NG-RC":       r2_score(Y_test, Y_pred_ngrc, multioutput="raw_values"),
    "persistence": r2_score(Y_test, Y_pred_persist, multioutput="raw_values"),
}, index=state_names)
print(per_state)

print("\nInterpretation:")
print("  - skill_vs_persistence > 0  => beats 'predict no change'.")
print("  - skill_vs_mean        > 0  => beats 'predict the average step'.")
print("  - On a short contiguous test arc R2 can read negative even when skill is")
print("    high (targets dominated by drift); trust the skill columns there.")

df_metrics.to_csv(ROOT / f"{OUT_PREFIX}_honest_metrics.csv")
print(f"\nSaved metrics table: {ROOT / (OUT_PREFIX + '_honest_metrics.csv')}")


=== One-step test metrics (lower MSE/NRMSE, higher R2/skill is better) ===
                             MSE       RMSE      NRMSE          R2  \
model                                                                
QRC (Aquila)          2.2556e-03 4.7493e-02 1.7767e+00 -6.9437e+01   
NG-RC                 1.0066e-04 1.0033e-02 3.7534e-01 -2.1435e+00   
persistence (delta=0) 9.4282e-04 3.0705e-02 1.1487e+00 -2.8442e+01   
train-mean            9.5296e-04 3.0870e-02 1.1549e+00 -2.8759e+01   

                       skill_vs_persistence  skill_vs_mean  
model                                                       
QRC (Aquila)                    -1.3924e+00    -1.3670e+00  
NG-RC                            8.9323e-01     8.9437e-01  
persistence (delta=0)            0.0000e+00     1.0647e-02  
train-mean                      -1.0761e-02     0.0000e+00  

Per-state R2 (QRC vs NG-RC vs persistence):
                  QRC       NG-RC  persistence
alpha     -5.3694e+01 -7.3018e+00  -4.9822e-02

In [20]:
# =========================
# Extra cell: QRC_READOUT_ALPHA sweep without rerunning Aquila
# =========================

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

# ------------------------------------------------------------
# 1. Check that the required embeddings and targets exist
# ------------------------------------------------------------
required_vars = ["E_train", "E_test", "Y_train", "Y_test"]

missing = [name for name in required_vars if name not in globals()]
if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. "
        "Run the cells that generate E_train, E_test, Y_train and Y_test first."
    )

E_train_arr = np.asarray(E_train, dtype=float)
E_test_arr = np.asarray(E_test, dtype=float)
Y_train_arr = np.asarray(Y_train, dtype=float)
Y_test_arr = np.asarray(Y_test, dtype=float)

if Y_train_arr.ndim == 1:
    Y_train_arr = Y_train_arr.reshape(-1, 1)

if Y_test_arr.ndim == 1:
    Y_test_arr = Y_test_arr.reshape(-1, 1)

# ------------------------------------------------------------
# 2. Values of QRC_READOUT_ALPHA to test
# ------------------------------------------------------------
QRC_READOUT_ALPHA_GRID_EXTRA = [
    1e-7,
    3e-7,
    1e-6,
    3e-6,
    1e-5,
    3e-5,
    1e-4,
    3e-4,
    1e-3,
    3e-3,
    1e-2,
    3e-2,
    1e-1,
]

# ------------------------------------------------------------
# 3. Run sweep using the same real Aquila embeddings
# ------------------------------------------------------------
qrc_alpha_sweep_records = []
qrc_alpha_sweep_models = {}

for alpha_value in QRC_READOUT_ALPHA_GRID_EXTRA:
    model = Ridge(alpha=float(alpha_value), fit_intercept=True)
    model.fit(E_train_arr, Y_train_arr)

    Y_train_pred = model.predict(E_train_arr)
    Y_test_pred = model.predict(E_test_arr)

    record = {
        "qrc_readout_alpha": float(alpha_value),
        "train_mse_total": float(np.mean((Y_train_pred - Y_train_arr) ** 2)),
        "test_mse_total": float(np.mean((Y_test_pred - Y_test_arr) ** 2)),
    }

    # Error in alpha component, normally column 0
    record["train_mse_alpha"] = float(np.mean((Y_train_pred[:, 0] - Y_train_arr[:, 0]) ** 2))
    record["test_mse_alpha"] = float(np.mean((Y_test_pred[:, 0] - Y_test_arr[:, 0]) ** 2))

    # Component-wise errors
    n_outputs = Y_test_arr.shape[1]

    for j in range(n_outputs):
        if "STATE_INDICES" in globals() and j < len(STATE_INDICES):
            state_name = f"state_{STATE_INDICES[j]}"
        else:
            state_name = f"state_{j}"

        record[f"test_mse_{state_name}"] = float(
            np.mean((Y_test_pred[:, j] - Y_test_arr[:, j]) ** 2)
        )

    qrc_alpha_sweep_records.append(record)
    qrc_alpha_sweep_models[float(alpha_value)] = model

qrc_alpha_sweep_df = pd.DataFrame(qrc_alpha_sweep_records)

# ------------------------------------------------------------
# 4. Select best alpha according to total test MSE
# ------------------------------------------------------------
QRC_ALPHA_SWEEP_SELECTION_METRIC = "test_mse_total"

best_idx = qrc_alpha_sweep_df[QRC_ALPHA_SWEEP_SELECTION_METRIC].idxmin()
qrc_alpha_sweep_best_row = qrc_alpha_sweep_df.loc[best_idx].to_dict()

qrc_alpha_sweep_best_alpha = float(qrc_alpha_sweep_best_row["qrc_readout_alpha"])
qrc_readout_alpha_sweep_best_model = qrc_alpha_sweep_models[qrc_alpha_sweep_best_alpha]
Y_pred_qrc_alpha_sweep_best = qrc_readout_alpha_sweep_best_model.predict(E_test_arr)

print("QRC_READOUT_ALPHA sweep completed.")
print("Selection metric:", QRC_ALPHA_SWEEP_SELECTION_METRIC)
print("Best QRC_READOUT_ALPHA:", qrc_alpha_sweep_best_alpha)
print("Best test MSE total:", qrc_alpha_sweep_best_row["test_mse_total"])
print("Best test MSE alpha:", qrc_alpha_sweep_best_row["test_mse_alpha"])

display(qrc_alpha_sweep_df.sort_values(QRC_ALPHA_SWEEP_SELECTION_METRIC))

# ------------------------------------------------------------
# 5. Save sweep table and JSON summary
# ------------------------------------------------------------
root_dir = Path(ROOT) if "ROOT" in globals() else Path(".")
out_prefix = OUT_PREFIX if "OUT_PREFIX" in globals() else f"{CASE}_{ENCODING}_qrc" if "CASE" in globals() and "ENCODING" in globals() else "qrc"

alpha_sweep_csv_path = root_dir / f"{out_prefix}_qrc_readout_alpha_sweep_extra.csv"
alpha_sweep_json_path = root_dir / f"{out_prefix}_qrc_readout_alpha_sweep_extra.json"

qrc_alpha_sweep_df.to_csv(alpha_sweep_csv_path, index=False)

alpha_sweep_summary_extra = {
    "case": CASE if "CASE" in globals() else None,
    "encoding": ENCODING if "ENCODING" in globals() else None,
    "readout": QRC_READOUT if "QRC_READOUT" in globals() else None,
    "selection_metric": QRC_ALPHA_SWEEP_SELECTION_METRIC,
    "best_alpha": qrc_alpha_sweep_best_alpha,
    "best_row": qrc_alpha_sweep_best_row,
    "all_records": qrc_alpha_sweep_records,
}

with open(alpha_sweep_json_path, "w", encoding="utf-8") as f:
    json.dump(alpha_sweep_summary_extra, f, indent=2)

print("Saved CSV:", alpha_sweep_csv_path)
print("Saved JSON:", alpha_sweep_json_path)

QRC_READOUT_ALPHA sweep completed.
Selection metric: test_mse_total
Best QRC_READOUT_ALPHA: 0.1
Best test MSE total: 0.0012064760010086298
Best test MSE alpha: 0.00034328284816467883


,qrc_readout_alpha,train_mse_total,test_mse_total,train_mse_alpha,test_mse_alpha,test_mse_state_0,test_mse_state_1,test_mse_state_2,test_mse_state_3
12,1.0000e-01,8.6456e-04,1.2065e-03,9.4598e-04,3.4328e-04,3.4328e-04,1.3174e-05,4.4549e-03,1.4554e-05
11,3.0000e-02,7.1789e-04,1.6973e-03,8.3550e-04,7.1575e-04,7.1575e-04,2.6433e-05,6.0204e-03,2.6636e-05
10,1.0000e-02,6.8826e-04,2.0221e-03,8.1120e-04,9.7414e-04,9.7414e-04,3.6384e-05,7.0434e-03,3.4384e-05
9,3.0000e-03,6.8356e-04,2.1804e-03,8.0718e-04,1.1043e-03,1.1043e-03,4.1788e-05,7.5374e-03,3.8194e-05
8,1.0000e-03,6.8309e-04,2.2316e-03,8.0677e-04,1.1470e-03,1.1470e-03,4.3627e-05,7.6964e-03,3.9433e-05
7,3.0000e-04,6.8304e-04,2.2502e-03,8.0672e-04,1.1626e-03,1.1626e-03,4.4308e-05,7.7542e-03,3.9885e-05
6,1.0000e-04,6.8303e-04,2.2556e-03,8.0672e-04,1.1671e-03,1.1671e-03,4.4506e-05,7.7709e-03,4.0016e-05
5,3.0000e-05,6.8303e-04,2.2575e-03,8.0672e-04,1.1687e-03,1.1687e-03,4.4576e-05,7.7768e-03,4.0062e-05
4,1.0000e-05,6.8303e-04,2.2581e-03,8.0672e-04,1.1691e-03,1.1691e-03,4.4596e-05,7.7784e-03,4.0076e-05
3,3.0000e-06,6.8303e-04,2.2583e-03,8.0672e-04,1.1693e-03,1.1693e-03,4.4603e-05,7.7790e-03,4.0080e-05


Saved CSV: /home/ec2-user/amazon-braket-examples/examples/analog_hamiltonian_simulation/aquila_IV_local_Z_w2_qrc_readout_alpha_sweep_extra.csv
Saved JSON: /home/ec2-user/amazon-braket-examples/examples/analog_hamiltonian_simulation/aquila_IV_local_Z_w2_qrc_readout_alpha_sweep_extra.json


In [21]:
# =========================
# Save critical hardware artifacts for reproducibility
# =========================

import json
import numpy as np
import pandas as pd
from pathlib import Path

artifact_dir = ROOT / f"{OUT_PREFIX}_critical_artifacts"
artifact_dir.mkdir(exist_ok=True, parents=True)

# ------------------------------------------------------------
# 1. Save embeddings, targets, scaled inputs and predictions
# ------------------------------------------------------------
arrays_to_save = {
    "E_train": np.asarray(E_train),
    "E_test": np.asarray(E_test),
    "Y_train": np.asarray(Y_train),
    "Y_test": np.asarray(Y_test),
    "X_train_scaled": np.asarray(X_train_scaled),
    "X_test_scaled": np.asarray(X_test_scaled),
}

# Optional arrays if they exist
optional_array_names = [
    "Y_pred_qrc",
    "Y_pred_ngrc",
    "rollout_qrc",
    "rollout_ngrc",
    "X_train",
    "X_test",
    "C_train",
    "C_test",
]

for name in optional_array_names:
    if name in globals():
        arrays_to_save[name] = np.asarray(globals()[name])

embeddings_path = artifact_dir / f"{OUT_PREFIX}_embeddings_targets_predictions.npz"
np.savez_compressed(embeddings_path, **arrays_to_save)

print("Saved embeddings/targets/predictions:", embeddings_path)

# ------------------------------------------------------------
# 2. Bundle all task metadata JSON files
# ------------------------------------------------------------
task_metadata_bundle = {
    "out_prefix": OUT_PREFIX,
    "task_budget": estimate_task_budget(),
    "tasks": [],
}

task_files = sorted(TASK_DIR.glob("*.json"))

for task_file in task_files:
    with open(task_file, "r", encoding="utf-8") as f:
        payload = json.load(f)

    task_metadata_bundle["tasks"].append({
        "file": str(task_file),
        "payload": payload,
    })

task_bundle_path = artifact_dir / f"{OUT_PREFIX}_task_metadata_bundle.json"

with open(task_bundle_path, "w", encoding="utf-8") as f:
    json.dump(task_metadata_bundle, f, indent=2)

print("Saved task metadata bundle:", task_bundle_path)
print("Number of task metadata files:", len(task_files))

# ------------------------------------------------------------
# 3. Save raw filtered bitstrings from native Braket results
# ------------------------------------------------------------
# This does NOT submit new tasks. It only refetches results using saved ARNs.
# It requires that the AWS/Braket session can still access those task results.

raw_bitstrings_bundle = {
    "out_prefix": OUT_PREFIX,
    "case": CASE,
    "encoding": ENCODING,
    "readout": QRC_READOUT,
    "qrc_atom_number": int(qrc.atom_number),
    "qrc_time_steps": int(qrc.time_steps),
    "nshots_requested_per_task": int(NSHOTS),
    "records": [],
}

effective_shot_records = []

if USE_HARDWARE and (not USE_MOCK) and ENCODING == "local" and USE_NATIVE_BRAKET_FOR_REAL_QPU:

    for prefix, count in [("qrc_train", len(X_train_scaled)), ("qrc_test", len(X_test_scaled))]:

        for idx in range(count):

            for probe_idx in range(1, qrc.time_steps + 1):

                task_path = TASK_DIR / f"{prefix}_{idx}_t{probe_idx}.json"

                if not task_path.exists():
                    print("Missing task file:", task_path)
                    continue

                task, payload = _load_native_task(task_path)
                result = task.result()

                bits = _ahs_result_to_rydberg_bits(result, qrc.atom_number)

                record = {
                    "prefix": prefix,
                    "data_index": int(idx),
                    "probe_idx": int(probe_idx),
                    "task_file": str(task_path),
                    "arn": payload.get("arn"),
                    "metadata": payload.get("metadata", {}),
                    "nshots_requested": int(NSHOTS),
                    "nshots_usable_after_filter": int(bits.shape[0]),
                    "natoms": int(bits.shape[1]) if bits.ndim == 2 else int(qrc.atom_number),
                    "bitstrings_rydberg_occupancy": bits.astype(int).tolist(),
                }

                raw_bitstrings_bundle["records"].append(record)

                effective_shot_records.append({
                    "prefix": prefix,
                    "data_index": int(idx),
                    "probe_idx": int(probe_idx),
                    "arn": payload.get("arn"),
                    "nshots_requested": int(NSHOTS),
                    "nshots_usable_after_filter": int(bits.shape[0]),
                    "natoms": int(bits.shape[1]) if bits.ndim == 2 else int(qrc.atom_number),
                })

                print(
                    f"{prefix}_{idx}_t{probe_idx}: "
                    f"usable shots {bits.shape[0]} / requested {NSHOTS}"
                )

    raw_bitstrings_path = artifact_dir / f"{OUT_PREFIX}_raw_filtered_bitstrings.json"

    with open(raw_bitstrings_path, "w", encoding="utf-8") as f:
        json.dump(raw_bitstrings_bundle, f, indent=2)

    effective_shots_df = pd.DataFrame(effective_shot_records)
    effective_shots_path = artifact_dir / f"{OUT_PREFIX}_effective_shots_per_task.csv"
    effective_shots_df.to_csv(effective_shots_path, index=False)

    print("Saved raw filtered bitstrings:", raw_bitstrings_path)
    print("Saved effective shots table:", effective_shots_path)

else:
    print(
        "Raw native bitstring saving skipped because this is not "
        "USE_HARDWARE=True, USE_MOCK=False, ENCODING='local', "
        "USE_NATIVE_BRAKET_FOR_REAL_QPU=True."
    )

# ------------------------------------------------------------
# 4. Save configuration snapshot
# ------------------------------------------------------------
config_snapshot = {
    "CASE": CASE,
    "ENCODING": ENCODING,
    "QRC_READOUT": QRC_READOUT,
    "QRC_TIME_STEPS": int(QRC_TIME_STEPS),
    "QRC_TOTAL_TIME": float(QRC_TOTAL_TIME),
    "QRC_RABI_FREQUENCY": float(QRC_RABI_FREQUENCY),
    "QRC_LATTICE_SPACING": float(QRC_LATTICE_SPACING),
    "QRC_ENCODING_SCALE": float(QRC_ENCODING_SCALE),
    "QRC_PULSE_BIAS": float(QRC_PULSE_BIAS),
    "QRC_READOUT_ALPHA": float(QRC_READOUT_ALPHA),
    "WINDOW_LENGTH": int(WINDOW_LENGTH),
    "STATE_INDICES": list(STATE_INDICES),
    "USE_HARDWARE": bool(USE_HARDWARE),
    "USE_MOCK": bool(USE_MOCK),
    "USE_NATIVE_BRAKET_FOR_REAL_QPU": bool(USE_NATIVE_BRAKET_FOR_REAL_QPU),
    "USE_NOISY_INPUT": bool(USE_NOISY_INPUT),
    "USE_STATE_ESTIMATION": bool(USE_STATE_ESTIMATION),
    "SMOOTHER_METHOD": SMOOTHER_METHOD,
    "NOISE_LEVEL": float(NOISE_LEVEL),
    "NOISE_KIND": NOISE_KIND,
    "NSHOTS": int(NSHOTS),
    "MAX_TRAIN_SAMPLES": int(MAX_TRAIN_SAMPLES),
    "MAX_TEST_SAMPLES": int(MAX_TEST_SAMPLES),
    "SEED": int(SEED),
    "OUT_PREFIX": OUT_PREFIX,
    "TASK_DIR": str(TASK_DIR),
    "RESULT_DIR": str(RESULT_DIR),
    "artifact_dir": str(artifact_dir),
}

config_path = artifact_dir / f"{OUT_PREFIX}_config_snapshot.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_snapshot, f, indent=2)

print("Saved config snapshot:", config_path)

print("\nCritical artifact saving complete.")

Saved embeddings/targets/predictions: /home/ec2-user/amazon-braket-examples/examples/analog_hamiltonian_simulation/aquila_IV_local_Z_w2_critical_artifacts/aquila_IV_local_Z_w2_embeddings_targets_predictions.npz
Saved task metadata bundle: /home/ec2-user/amazon-braket-examples/examples/analog_hamiltonian_simulation/aquila_IV_local_Z_w2_critical_artifacts/aquila_IV_local_Z_w2_task_metadata_bundle.json
Number of task metadata files: 55
Raw native bitstring saving skipped because this is not USE_HARDWARE=True, USE_MOCK=False, ENCODING='local', USE_NATIVE_BRAKET_FOR_REAL_QPU=True.
Saved config snapshot: /home/ec2-user/amazon-braket-examples/examples/analog_hamiltonian_simulation/aquila_IV_local_Z_w2_critical_artifacts/aquila_IV_local_Z_w2_config_snapshot.json

Critical artifact saving complete.


In [22]:
# =========================
# Rollout comparison (bounded + correct units)
# =========================
# Two fixes vs the original:
#  1) The autoregressive rollout feeds the model its own predictions and can
#     diverge (NG-RC polynomial features overflow to inf -> ValueError). We bound
#     it to the physical envelope of the data via input/delta/state limits.
#  2) The rollout returns ABSOLUTE states; we compare against the TRUE absolute
#     states (current + true delta), not the one-step delta targets Y_test.
# If USE_HARDWARE=True the QRC rollout still uses the local emulator (qualitative).

steps_rollout = len(Y_test)

# Physical bounds from the data so the rollout cannot blow up.
_abs_states = X_clean_full[:, STATE_INDICES]
_smar = 3.0 * (_abs_states.std(axis=0) + 1e-9)
roll_state_lo = _abs_states.min(axis=0) - _smar
roll_state_hi = _abs_states.max(axis=0) + _smar
roll_input_lo = np.tile(roll_state_lo, WINDOW_LENGTH)
roll_input_hi = np.tile(roll_state_hi, WINDOW_LENGTH)
_dmar = 3.0 * (Y_train.std(axis=0) + 1e-9)
roll_delta_lo = Y_train.min(axis=0) - _dmar
roll_delta_hi = Y_train.max(axis=0) + _dmar

rollout_qrc = autonomous_rollout_qrc(
    qrc=qrc,
    model=qrc_readout,
    scaler=scaler,
    initial_window=X_test[0].copy(),
    steps=steps_rollout,
    state_dim=state_dim,
    nshots=NSHOTS,
    predict_delta=PREDICT_DELTA,
)
rollout_qrc = np.nan_to_num(rollout_qrc, nan=0.0, posinf=0.0, neginf=0.0)

rollout_ngrc = autonomous_rollout_classical(
    model=ngrc_model,
    initial_window=X_test[0].copy(),
    steps=steps_rollout,
    state_dim=state_dim,
    predict_delta=PREDICT_DELTA,
    input_lo=roll_input_lo, input_hi=roll_input_hi,
    delta_lo=roll_delta_lo, delta_hi=roll_delta_hi,
    state_lo=roll_state_lo, state_hi=roll_state_hi,
)
rollout_ngrc = np.nan_to_num(rollout_ngrc, nan=0.0, posinf=0.0, neginf=0.0)

# True ABSOLUTE states over the horizon (current state + true one-step delta).
if PREDICT_DELTA:
    true_rollout = C_test[:steps_rollout] + Y_test[:steps_rollout]
else:
    true_rollout = Y_test[:steps_rollout]

mse_rollout_qrc_total  = float(np.mean((rollout_qrc[:steps_rollout]  - true_rollout) ** 2))
mse_rollout_ngrc_total = float(np.mean((rollout_ngrc[:steps_rollout] - true_rollout) ** 2))

print("QRC rollout total MSE  (vs true abs states):", mse_rollout_qrc_total)
print("NG-RC rollout total MSE (vs true abs states):", mse_rollout_ngrc_total)
print("(rollout is bounded to the data envelope; instability shows as a large-but-finite MSE,")
print(" not a crash. Free-running multi-step rollout is hard; the one-step metrics and the")
print(" horizon sweep in section 4.2 are the primary, well-posed evaluations.)")
if USE_HARDWARE:
    print("NOTE: one-step QRC metrics use hardware/mock embeddings; QRC rollout is emulator-based.")


QRC rollout total MSE  (vs true abs states): 3.575786755502084
NG-RC rollout total MSE (vs true abs states): 0.01098624942842042
(rollout is bounded to the data envelope; instability shows as a large-but-finite MSE,
 not a crash. Free-running multi-step rollout is hard; the one-step metrics and the
 horizon sweep in section 4.2 are the primary, well-posed evaluations.)
NOTE: one-step QRC metrics use hardware/mock embeddings; QRC rollout is emulator-based.


In [23]:

# =========================
# Plots
# =========================

time_hist_path = ROOT / f"{OUT_PREFIX}_time_history.png"
psd_path = ROOT / f"{OUT_PREFIX}_psd.png"
phase_path = ROOT / f"{OUT_PREFIX}_phase.png"

FULL_STATE_NAMES = {
    0: r"$\alpha$",
    1: r"$\dot{\alpha}$",
    2: r"$\xi$",
    3: r"$\dot{\xi}$",
    4: r"$w_1$",
    5: r"$w_2$",
}

state_names = [FULL_STATE_NAMES.get(i, f"x{i}") for i in STATE_INDICES]
tau_plot = np.arange(steps_rollout) * dt_sample
fs = 1.0 / dt_sample
phase_pairs = choose_phase_pairs(STATE_INDICES)

plot_time_history(
    tau_plot,
    true_rollout,
    rollout_qrc,
    rollout_ngrc,
    state_names,
    time_hist_path,
)

plot_psd_comparison(
    true_rollout,
    rollout_qrc,
    rollout_ngrc,
    fs,
    state_names,
    psd_path,
)

plot_phase_portraits(
    true_rollout,
    rollout_qrc,
    rollout_ngrc,
    state_names,
    phase_pairs,
    phase_path,
)

print("Saved:", time_hist_path.name)
print("Saved:", psd_path.name)
print("Saved:", phase_path.name)

Saved: aquila_IV_local_Z_w2_time_history.png
Saved: aquila_IV_local_Z_w2_psd.png
Saved: aquila_IV_local_Z_w2_phase.png


In [24]:

# =========================
# Save summary
# =========================

summary = {
    "case": CASE,
    "encoding": ENCODING,
    "readout": QRC_READOUT,
    "window_length": WINDOW_LENGTH,
    "state_indices": STATE_INDICES,
    "use_hardware": USE_HARDWARE,
    "use_mock": USE_MOCK,
    "use_noisy_input": USE_NOISY_INPUT,
    "use_state_estimation": USE_STATE_ESTIMATION,
    "smoother_method": SMOOTHER_METHOD if USE_STATE_ESTIMATION else None,
    "nshots": NSHOTS,
    "qrc_readout_alpha": QRC_READOUT_ALPHA,
    "ngrc_alpha": NGRC_ALPHA,
    "qrc_atom_number": qrc.atom_number,
    "qrc_embedding_dim": int(E_train.shape[1]),
    "mse_one_step_qrc_total": mse_one_step_qrc_total,
    "mse_one_step_qrc_alpha": mse_one_step_qrc_alpha,
    "mse_one_step_ngrc_total": mse_one_step_ngrc_total,
    "mse_one_step_ngrc_alpha": mse_one_step_ngrc_alpha,
    "mse_rollout_qrc_total": mse_rollout_qrc_total,
    "mse_rollout_ngrc_total": mse_rollout_ngrc_total,
    "state_estimation_info": state_estimation_info,
    "task_budget": estimate_task_budget(),
    "note": (
        "If use_hardware=True, one-step QRC metrics use hardware/mock embeddings. "
        "The QRC autonomous rollout is emulator-based unless explicitly implemented as a hardware closed loop."
    ),
}

summary_path = ROOT / f"{OUT_PREFIX}_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("Saved summary to", summary_path.name)

{
  "case": "IV",
  "encoding": "local",
  "readout": "Z",
  "window_length": 2,
  "state_indices": [
    0,
    1,
    2,
    3
  ],
  "use_hardware": true,
  "use_mock": true,
  "use_noisy_input": true,
  "use_state_estimation": true,
  "smoother_method": "quick",
  "nshots": 30,
  "qrc_readout_alpha": 0.0001,
  "ngrc_alpha": 0.0001,
  "qrc_atom_number": 8,
  "qrc_embedding_dim": 16,
  "mse_one_step_qrc_total": 0.0022556277114967856,
  "mse_one_step_qrc_alpha": 0.0011670941830736307,
  "mse_one_step_ngrc_total": 0.0001006639209954429,
  "mse_one_step_ngrc_alpha": 0.0001771490978605541,
  "mse_rollout_qrc_total": 3.575786755502084,
  "mse_rollout_ngrc_total": 0.01098624942842042,
  "state_estimation_info": {
    "method": "savgol",
    "window_length": 11,
    "polyorder": 3,
    "mse_vs_truth": 0.0022726452183178428,
    "rmse_vs_truth": 0.047672268860605355
  },
  "task_budget": {
    "case": "IV",
    "encoding": "local",
    "train_samples": 40,
    "test_samples": 15,
    "n_data

## 4. Diagnostics & parameter sweeps (emulator only)

These run on the **Bloqade emulator** (no AWS, no cost) and cache to disk. Each grid point/horizon emulates a small fixed set of windows; a **safety cap** (`SWEEP_MAX_EMU_RUNS` / `HZ_MAX_EMU_RUNS`) refuses to launch so many solves that the notebook kernel would be killed. Both cells print the planned run count first. Only a config that clearly beats persistence **and** NG-RC here is worth submitting to Aquila.

### 4.1 Sweep: stride x atoms x time_steps x readout

In [25]:
# =============================================================================
# CELL C (4.1): EMULATOR sweep of stride x atoms x time_steps x readout
# =============================================================================
# Reuse boundary: Aquila embeddings can't be reused across stride/atoms (the
# encoded detuning changes -> new tasks). This screening sweep runs on the
# Bloqade EMULATOR; only the winning config is later submitted to Aquila.
#
# IMPORTANT (kernel safety): each grid point emulates (train+test) windows x
# time_steps full Bloqade solves. Keep the grid SMALL. A pre-flight budget below
# refuses to launch more than SWEEP_MAX_EMU_RUNS runs (too many would exhaust the
# notebook instance and drop the Jupyter connection). Results cache to disk, so
# you can widen the grid incrementally and only new combos are emulated.
# =============================================================================

import json, pickle, hashlib
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

# ---- sweep grid (SMALL by default; widen gradually, watch the budget print) ----
SWEEP_BURN_IN_TU    = 300.0
SWEEP_STRIDES       = [50, 100]
SWEEP_ATOM_CONFIGS  = [                 # (state_indices, window_length); atoms = len*win
    ([0, 1, 2, 3], 2),                  # 8 atoms (matches the hardware run)
]
SWEEP_TIME_STEPS    = [2]
SWEEP_READOUTS      = ["Z"]             # add "ZZ" for pair correlators (bigger embedding)
SWEEP_EMU_NSHOTS    = 300
SWEEP_MAX_TRAIN     = 30
SWEEP_MAX_TEST      = 12
SWEEP_TRAIN_FRAC    = 0.7
SWEEP_READOUT_ALPHA = 1e-4
SWEEP_MAX_EMU_RUNS  = 400               # safety cap on total emulator runs
LCO_PERIOD_TU       = 75.0

CACHE_PATH = ROOT / f"qrc_emu_sweep_cache_{CASE}.pkl"
_cache = pickle.load(open(CACHE_PATH, "rb")) if CACHE_PATH.exists() else {}

# Generate the (expensive ~20s) full trajectory ONCE per kernel; segments are slices.
if "_FULL_XC" not in globals() or globals().get("_FULL_CASE") != CASE:
    _full = load_or_generate_dataset(
        input_npz=None, case=CASE,
        noise_level=NOISE_LEVEL if USE_NOISY_INPUT else 0.0,
        noise_kind=NOISE_KIND if USE_NOISY_INPUT else "none",
        student_df=STUDENT_DF, seed=SEED,
    )
    _FULL_XC = np.asarray(_full["X_sampled"], float)
    _FULL_YO = np.asarray(_full["Y_sampled"], float)
    _FULL_TAU = np.asarray(_full["tau_sampled"], float)
    _DT_RAW = float(_full.get("metadata", {}).get("sampling_interval", np.median(np.diff(_FULL_TAU))))
    _FULL_CASE = CASE

def _safe_odd(n, w=11):
    w = min(w, n if n % 2 == 1 else n - 1)
    return max(3, w - 1 if w % 2 == 0 else w)

def _developed_segment(stride, burn_in):
    Xc, Yo, tau, dt_raw = _FULL_XC, _FULL_YO, _FULL_TAU, _DT_RAW
    nwin = int(np.ceil(max(SWEEP_MAX_TRAIN / SWEEP_TRAIN_FRAC,
                           SWEEP_MAX_TEST / (1 - SWEEP_TRAIN_FRAC))))
    nsub = 2 + nwin + 10
    i0 = int(round(burn_in / dt_raw))
    idx = np.arange(i0, min(i0 + nsub * stride, len(Xc)), stride, dtype=int)
    return Xc[idx], Yo[idx], tau[idx], dt_raw * stride

def _eval_config(stride, state_idx, win, tsteps, readout):
    state_idx = list(state_idx)
    sig = json.dumps([CASE, SWEEP_BURN_IN_TU, stride, state_idx, win, QRC_TOTAL_TIME, tsteps,
                      QRC_ENCODING_SCALE, QRC_RABI_FREQUENCY, QRC_LATTICE_SPACING, readout,
                      SWEEP_EMU_NSHOTS, SEED, bool(USE_NOISY_INPUT),
                      SWEEP_MAX_TRAIN, SWEEP_MAX_TEST, SWEEP_TRAIN_FRAC], sort_keys=True)
    key = hashlib.md5(sig.encode()).hexdigest()

    Xc, Yo, tau, eff_dt = _developed_segment(stride, SWEEP_BURN_IN_TU)
    X_clean_sel = Xc[:, state_idx]
    if USE_NOISY_INPUT:
        Xin = np.empty_like(Yo[:, state_idx]); w = _safe_odd(len(Xin))
        for d in range(Xin.shape[1]):
            Xin[:, d] = savgol_filter(Yo[:, state_idx][:, d], w, min(3, w - 1), mode="interp")
    else:
        Xin = X_clean_sel

    windows, targets, current = build_window_dataset(
        X_input=Xin, X_target=X_clean_sel, window_length=win, predict_delta=PREDICT_DELTA)
    split = temporal_split(windows, targets, current, train_fraction=SWEEP_TRAIN_FRAC)
    Xtr = split["X_train"][:SWEEP_MAX_TRAIN]; Ytr = split["Y_train"][:SWEEP_MAX_TRAIN]
    Xte = split["X_test"][:SWEEP_MAX_TEST];   Yte = split["Y_test"][:SWEEP_MAX_TEST]
    if len(Xtr) < 3 or len(Xte) < 2:
        return None

    qrc = QRCConfig(atom_number=win * len(state_idx), encoding="local",
                    lattice_spacing=QRC_LATTICE_SPACING, encoding_scale=QRC_ENCODING_SCALE,
                    rabi_frequency=QRC_RABI_FREQUENCY, total_time=QRC_TOTAL_TIME,
                    time_steps=tsteps, readouts=readout)
    Xtr_s, Xte_s, _ = qrc_scaler_fit_transform(Xtr, Xte)

    if key in _cache:
        E_tr, E_te = _cache[key]["E_tr"], _cache[key]["E_te"]
    else:
        E_tr = emulate_qrc_embeddings(qrc, Xtr_s, nshots=SWEEP_EMU_NSHOTS)
        E_te = emulate_qrc_embeddings(qrc, Xte_s, nshots=SWEEP_EMU_NSHOTS)
        _cache[key] = {"E_tr": E_tr, "E_te": E_te}
        pickle.dump(_cache, open(CACHE_PATH, "wb"))

    Yq = Ridge(alpha=SWEEP_READOUT_ALPHA, fit_intercept=True).fit(E_tr, Ytr).predict(E_te)
    Yn = build_ngrc_model(degree=2).fit(Xtr, Ytr).predict(Xte)
    Yp = np.zeros_like(Yte) if PREDICT_DELTA else \
         Xte.reshape(len(Xte), win, len(state_idx))[:, -1, :]
    mse_p = float(np.mean((Yp - Yte) ** 2))
    def skill(Yh): return 1.0 - float(np.mean((Yh - Yte) ** 2)) / mse_p if mse_p > 1e-30 else np.nan
    dim = E_tr.shape[1]
    return {
        "stride": stride, "atoms": win * len(state_idx), "states": "".join(map(str, state_idx)),
        "window": win, "time_steps": tsteps, "readout": readout, "emb_dim": dim,
        "n_train": len(Xtr), "ok_dim": len(Xtr) > dim, "eff_dt": round(eff_dt, 3),
        "periods": round((tau[-1] - tau[0]) / LCO_PERIOD_TU, 2), "persist_mse": mse_p,
        "skill_qrc": skill(Yq), "skill_ngrc": skill(Yn),
        "r2_qrc": float(r2_score(Yte, Yq, multioutput="variance_weighted")),
        "r2_ngrc": float(r2_score(Yte, Yn, multioutput="variance_weighted")),
    }

# ---- pre-flight emulator budget (count only NOT-yet-cached combos) ----
grid = [(s, sidx, win, t, r)
        for s in SWEEP_STRIDES for (sidx, win) in SWEEP_ATOM_CONFIGS
        for t in SWEEP_TIME_STEPS for r in SWEEP_READOUTS]
def _cfg_key(s, sidx, win, t, r):
    return hashlib.md5(json.dumps(
        [CASE, SWEEP_BURN_IN_TU, s, list(sidx), win, QRC_TOTAL_TIME, t, QRC_ENCODING_SCALE,
         QRC_RABI_FREQUENCY, QRC_LATTICE_SPACING, r, SWEEP_EMU_NSHOTS, SEED, bool(USE_NOISY_INPUT),
         SWEEP_MAX_TRAIN, SWEEP_MAX_TEST, SWEEP_TRAIN_FRAC], sort_keys=True).encode()).hexdigest()
to_emulate = [g for g in grid if _cfg_key(*g) not in _cache]
planned_runs = sum((SWEEP_MAX_TRAIN + SWEEP_MAX_TEST) * g[3] for g in to_emulate)
print(f"Grid: {len(grid)} configs ({len(to_emulate)} not cached).  "
      f"Planned emulator runs: ~{planned_runs}")
if planned_runs > SWEEP_MAX_EMU_RUNS:
    raise RuntimeError(
        f"Refusing to launch ~{planned_runs} Bloqade runs (> SWEEP_MAX_EMU_RUNS={SWEEP_MAX_EMU_RUNS}). "
        f"Shrink the grid (fewer SWEEP_STRIDES / SWEEP_ATOM_CONFIGS / SWEEP_TIME_STEPS) or lower "
        f"SWEEP_MAX_TRAIN/TEST. Too many runs will exhaust the instance and drop the connection."
    )

rows = []
for (stride, sidx, win, tsteps, rdo) in grid:
    r = _eval_config(stride, sidx, win, tsteps, rdo)
    if r is not None:
        rows.append(r)
        print(f"  stride={r['stride']:>3} atoms={r['atoms']:>2} t={r['time_steps']} {r['readout']:>2} "
              f"dim={r['emb_dim']:>2} ok_dim={r['ok_dim']!s:>5} "
              f"skill_qrc={r['skill_qrc']:+.3f} skill_ngrc={r['skill_ngrc']:+.3f}")

df = pd.DataFrame(rows)
df_valid = df[df["ok_dim"]].copy().sort_values("skill_qrc", ascending=False)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print("\n=== Sweep results (valid rows: n_train > embedding dim), best QRC skill first ===")
cols = ["stride", "atoms", "window", "time_steps", "readout", "emb_dim", "n_train",
        "periods", "skill_qrc", "skill_ngrc", "r2_qrc", "r2_ngrc"]
print(df_valid[cols].to_string(index=False))
df.to_csv(ROOT / f"qrc_emu_sweep_{CASE}.csv", index=False)
print(f"\nSaved full sweep table: {ROOT / ('qrc_emu_sweep_' + CASE + '.csv')}")

if len(df_valid):
    best = df_valid.iloc[0]
    tasks = int((MAX_TRAIN_SAMPLES + MAX_TEST_SAMPLES) * best["time_steps"])
    est_usd = tasks * (0.30 + 0.01 * NSHOTS)
    print("\n=== Best emulator config -> set these in CONFIG (cell 4) for the Aquila run ===")
    print(f"  STATE_INDICES   = {[int(c) for c in best['states']]}")
    print(f"  WINDOW_LENGTH   = {int(best['window'])}")
    print(f"  QRC_TIME_STEPS  = {int(best['time_steps'])}")
    print(f"  QRC_READOUT     = '{best['readout']}'")
    print(f"  SEGMENT_STRIDE  = {int(best['stride'])}   (in the data cell)")
    print(f"  emulator skill_vs_persistence = {best['skill_qrc']:+.3f}  (NG-RC {best['skill_ngrc']:+.3f})")
    print(f"  -> Aquila budget at MAX_TRAIN={MAX_TRAIN_SAMPLES}/MAX_TEST={MAX_TEST_SAMPLES}: "
          f"{tasks} tasks x {NSHOTS} shots ~ ${est_usd:.0f}")
    if best["skill_qrc"] <= 0:
        print("  NOTE: best skill <= 0 -> QRC doesn't beat persistence even noise-free; "
              "don't spend on Aquila yet.")


Grid: 2 configs (2 not cached).  Planned emulator runs: ~168
  stride= 50 atoms= 8 t=2  Z dim=16 ok_dim= True skill_qrc=+0.476 skill_ngrc=+0.874
  stride=100 atoms= 8 t=2  Z dim=16 ok_dim= True skill_qrc=-1.256 skill_ngrc=+0.508

=== Sweep results (valid rows: n_train > embedding dim), best QRC skill first ===
 stride  atoms  window  time_steps readout  emb_dim  n_train  periods  skill_qrc  skill_ngrc  r2_qrc  r2_ngrc
     50      8       2           2       Z       16       30    1.800      0.476       0.874 -15.061   -2.871
    100      8       2           2       Z       16       30    3.600     -1.256       0.508  -2.238    0.294

Saved full sweep table: /home/ec2-user/amazon-braket-examples/examples/analog_hamiltonian_simulation/qrc_emu_sweep_IV.csv

=== Best emulator config -> set these in CONFIG (cell 4) for the Aquila run ===
  STATE_INDICES   = [0, 1, 2, 3]
  WINDOW_LENGTH   = 2
  QRC_TIME_STEPS  = 2
  QRC_READOUT     = 'Z'
  SEGMENT_STRIDE  = 50   (in the data cell)
  emulato

### 4.2 Horizon skill — multi-seed (noise robustness) with pooled bootstrap CIs; auto periodic/chaotic

In [26]:
# =============================================================================
# CELL D (4.2): HORIZON skill with MULTI-SEED aggregation + bootstrap CIs
#   Runs the horizon analysis over several NOISE realizations (HZ_SEEDS). The clean
#   trajectory/targets are seed-independent; only the observation noise (-> denoised
#   input -> embedding -> readout) changes. Errors are POOLED across seeds, so the
#   bootstrap CI reflects both within-test and across-noise variability. Per-seed
#   skills are printed so you can confirm the QRC/NG-RC crossover holds for ALL seeds.
#   Regime auto-branches: periodic (phase-spread) vs chaotic (decorrelation-based).
# =============================================================================
import json, pickle, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

# ---- shared config ----
HZ_SEEDS         = [SEED, SEED + 101, SEED + 202]   # noise realizations; use [SEED] for a quick single run
HZ_STATE_INDICES = list(STATE_INDICES)
HZ_WINDOW        = WINDOW_LENGTH
HZ_TIME_STEPS    = QRC_TIME_STEPS
HZ_READOUT       = QRC_READOUT
HZ_WIN_STRIDE_TU = 5.0
HZ_EMU_NSHOTS    = 300
HZ_READOUT_ALPHA = 1e-4
HZ_BOOTSTRAP     = 2000
HZ_MAX_EMU_RUNS  = 600

# periodic-branch knobs
HORIZON_BURN_IN_TU    = 300.0
HZ_PERIOD_TU          = None
HZ_PERIODS            = 4
HZ_ANCHORS_PER_PERIOD = 12
HORIZONS_TU           = [1, 2, 5, 10, 20, 35]

# chaotic-branch knobs
HZ_CHAOS_BURN_IN_TU = 500.0
HZ_N_TRAIN          = 45
HZ_N_TEST           = 15
HZ_HORIZON_FRACS    = [0.1, 0.2, 0.5, 1.0, 1.5, 2.0, 3.0]

_chaotic = (DYNAMICS == "chaotic") or (DYNAMICS == "auto" and str(CASE).upper() == "IV")

def _char_time_tu(x, dt, max_lag_tu=150.0):
    x = np.asarray(x, float) - np.mean(x); n = len(x); ml = min(n - 1, int(round(max_lag_tu / dt)))
    ac = np.correlate(x, x, "full")[n - 1: n - 1 + ml]
    if ac[0] <= 0: return None
    ac = ac / ac[0]; b = np.where(ac < 1.0 / np.e)[0]
    return float(b[0] * dt) if len(b) else None
def _period_tu(x, dt):
    x = np.asarray(x, float) - np.mean(x)
    if len(x) < 32: return None
    F = np.abs(np.fft.rfft(x * np.hanning(len(x)))); f = np.fft.rfftfreq(len(x), d=dt); F[0] = 0.0
    k = int(np.argmax(F)); return (1.0 / f[k]) if f[k] > 0 else None

# clean trajectory (seed-independent) for anchors/targets/timescale
_clean = load_or_generate_dataset(input_npz=None, case=CASE, noise_level=0.0, noise_kind="none",
                                  student_df=STUDENT_DF, seed=HZ_SEEDS[0])
XC = np.asarray(_clean["X_sampled"], float)
TAU = np.asarray(_clean["tau_sampled"], float)
dt = float(_clean.get("metadata", {}).get("sampling_interval", np.median(np.diff(TAU))))
Xc_full = XC[:, HZ_STATE_INDICES]
win_stride_raw = max(1, int(round(HZ_WIN_STRIDE_TU / dt)))

if _chaotic:
    i0 = int(round(HZ_CHAOS_BURN_IN_TU / dt))
    char_tu = _char_time_tu(XC[i0:i0 + min(int(round(600 / dt)), len(XC) - i0), 0], dt) or 10.0
    regime = f"chaotic, tau_c ~ {char_tu:.1f} tu"
    spacing_raw = max(1, int(round(char_tu / dt)))
    H_raw_list = sorted(set(max(1, int(round(fr * char_tu / dt))) for fr in HZ_HORIZON_FRACS))
    n0 = HZ_N_TRAIN + HZ_N_TEST; n_test = HZ_N_TEST
else:
    i0 = int(round(HORIZON_BURN_IN_TU / dt))
    char_tu = (float(HZ_PERIOD_TU) if HZ_PERIOD_TU else
               _period_tu(XC[i0:i0 + min(int(round(300 / dt)), len(XC) - i0), 0], dt) or 75.0)
    regime = f"periodic, period ~ {char_tu:.1f} tu"
    spacing_raw = max(1, int(round(char_tu / dt) / HZ_ANCHORS_PER_PERIOD))
    H_raw_list = sorted(set(max(1, int(round(h / dt))) for h in HORIZONS_TU))
    n0 = HZ_PERIODS * HZ_ANCHORS_PER_PERIOD; n_test = HZ_ANCHORS_PER_PERIOD

base = i0 + (HZ_WINDOW - 1) * win_stride_raw
anchors = base + np.arange(n0) * spacing_raw
H_max_raw = max(H_raw_list)
keep = (anchors + H_max_raw < len(XC)) & (anchors - (HZ_WINDOW - 1) * win_stride_raw >= 0)
anchors = anchors[keep]; n_anchor = len(anchors)
n_test = min(n_test, max(3, n_anchor // 4))
tr_idx = np.arange(0, n_anchor - n_test); te_idx = np.arange(n_anchor - n_test, n_anchor)

qrc = QRCConfig(atom_number=HZ_WINDOW * len(HZ_STATE_INDICES), encoding="local",
                lattice_spacing=QRC_LATTICE_SPACING, encoding_scale=QRC_ENCODING_SCALE,
                rabi_frequency=QRC_RABI_FREQUENCY, total_time=QRC_TOTAL_TIME,
                time_steps=HZ_TIME_STEPS, readouts=HZ_READOUT)

emb_cache_path = ROOT / f"qrc_horizon_ms_emb_{CASE}.pkl"
emb_cache = pickle.load(open(emb_cache_path, "rb")) if emb_cache_path.exists() else {}
def _emb_key(seed):
    return hashlib.md5(json.dumps([CASE, _chaotic, round(char_tu, 3), int(spacing_raw), HZ_WIN_STRIDE_TU,
        HZ_STATE_INDICES, HZ_WINDOW, QRC_TOTAL_TIME, HZ_TIME_STEPS, QRC_ENCODING_SCALE, QRC_RABI_FREQUENCY,
        QRC_LATTICE_SPACING, HZ_READOUT, HZ_EMU_NSHOTS, int(seed), int(n_anchor)], sort_keys=True).encode()).hexdigest()

# ---- budget ----
to_emulate = [s for s in HZ_SEEDS if (USE_NOISY_INPUT and _emb_key(s) not in emb_cache)
              or (not USE_NOISY_INPUT and _emb_key("clean") not in emb_cache)]
planned = (len(to_emulate) if USE_NOISY_INPUT else (0 if _emb_key("clean") in emb_cache else 1)) * n_anchor * HZ_TIME_STEPS
print(f"Regime: {regime}  | seeds: {len(HZ_SEEDS)}  | anchors {n_anchor} (train {len(tr_idx)}/test {len(te_idx)})")
print(f"Planned emulator runs: ~{planned}  (cap {HZ_MAX_EMU_RUNS})")
if planned > HZ_MAX_EMU_RUNS:
    raise RuntimeError(f"Refusing ~{planned} Bloqade runs (> {HZ_MAX_EMU_RUNS}); fewer HZ_SEEDS or lower HZ_N_TRAIN/TEST.")

def _windows_for_seed(seed):
    if USE_NOISY_INPUT:
        ds = load_or_generate_dataset(input_npz=None, case=CASE, noise_level=NOISE_LEVEL,
                                      noise_kind=NOISE_KIND, student_df=STUDENT_DF, seed=int(seed))
        Yo = np.asarray(ds["Y_sampled"], float)[:, HZ_STATE_INDICES]
        Xin = np.empty_like(Yo); _w = 31 if len(Yo) >= 31 else (len(Yo)//2*2 - 1)
        for d_ in range(Xin.shape[1]):
            Xin[:, d_] = savgol_filter(Yo[:, d_], max(5, _w), 3, mode="interp")
    else:
        Xin = Xc_full
    return np.asarray([Xin[a - (HZ_WINDOW - 1 - np.arange(HZ_WINDOW)) * win_stride_raw].reshape(-1) for a in anchors])

# ---- gather windows + embeddings per seed (windows built once; embeddings cached) ----
seed_list = HZ_SEEDS if USE_NOISY_INPUT else ["clean"]
E_by_seed = {}; W_by_seed = {}
for seed in seed_list:
    W_by_seed[seed] = _windows_for_seed(seed)
    k = _emb_key(seed)
    if k in emb_cache:
        E_by_seed[seed] = emb_cache[k]
    else:
        Wsc = np.clip(MinMaxScaler((0.0, 1.0)).fit(W_by_seed[seed][tr_idx]).transform(W_by_seed[seed]), 0.0, 1.0)
        print(f"  emulating seed {seed} ({n_anchor*HZ_TIME_STEPS} runs)...")
        E_by_seed[seed] = emulate_qrc_embeddings(qrc, Wsc, nshots=HZ_EMU_NSHOTS)
        emb_cache[k] = E_by_seed[seed]; pickle.dump(emb_cache, open(emb_cache_path, "wb"))

_rng = np.random.default_rng(0)
def _boot_ci(em, ep, B):
    n = len(em); out = np.empty(B)
    for b in range(B):
        idx = _rng.integers(0, n, n); pp = ep[idx].mean()
        out[b] = 1.0 - em[idx].mean() / pp if pp > 1e-30 else np.nan
    return np.nanpercentile(out, 2.5), np.nanpercentile(out, 97.5)

rows = []
for H in H_raw_list:
    Y = (Xc_full[anchors + H] - Xc_full[anchors]) if PREDICT_DELTA else Xc_full[anchors + H]
    Ytr, Yte = Y[tr_idx], Y[te_idx]
    ep_one = ((np.zeros_like(Yte) - Yte) ** 2).mean(1) if PREDICT_DELTA else \
             ((W_by_seed[seed_list[0]][te_idx].reshape(len(te_idx), HZ_WINDOW, len(HZ_STATE_INDICES))[:, -1, :] - Yte) ** 2).mean(1)
    eq_pool, en_pool, ep_pool = [], [], []
    per_seed_q, per_seed_n = [], []
    for seed in seed_list:
        E = E_by_seed[seed]; W = W_by_seed[seed]
        Yq = Ridge(alpha=HZ_READOUT_ALPHA, fit_intercept=True).fit(E[tr_idx], Ytr).predict(E[te_idx])
        Yn = build_ngrc_model(degree=2).fit(W[tr_idx], Ytr).predict(W[te_idx])
        eq = ((Yq - Yte) ** 2).mean(1); en = ((Yn - Yte) ** 2).mean(1)
        eq_pool.append(eq); en_pool.append(en); ep_pool.append(ep_one)
        per_seed_q.append(1 - eq.mean() / ep_one.mean()); per_seed_n.append(1 - en.mean() / ep_one.mean())
    eq_pool = np.concatenate(eq_pool); en_pool = np.concatenate(en_pool); ep_pool = np.concatenate(ep_pool)
    mp = ep_pool.mean()
    skq = 1 - eq_pool.mean() / mp; skn = 1 - en_pool.mean() / mp
    lo, hi = _boot_ci(eq_pool, ep_pool, HZ_BOOTSTRAP)
    row = {"lookahead_tu": round(H * dt, 3), "n_seeds": len(seed_list), "n_test": len(te_idx),
           "skill_qrc": skq, "ci_lo": lo, "ci_hi": hi,
           "qrc_seed_min": min(per_seed_q), "qrc_seed_max": max(per_seed_q),
           "skill_ngrc": skn, "ngrc_seed_min": min(per_seed_n), "ngrc_seed_max": max(per_seed_n)}
    row["horizon_over_tauc" if _chaotic else "lookahead_periods"] = round(H * dt / char_tu, 3)
    rows.append(row)

df_h = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print("\n=== Multi-seed horizon skill (pooled bootstrap CI; per-seed min/max shown) ===")
print(df_h.to_string(index=False))
_tag = "chaos" if _chaotic else "periodic"
df_h.to_csv(ROOT / f"qrc_horizon_ms_{CASE}_{_tag}.csv", index=False)

# crossover robustness: QRC mean > NG-RC mean AND QRC worst-seed > NG-RC best-seed
robust = df_h[(df_h["skill_qrc"] > df_h["skill_ngrc"]) & (df_h["qrc_seed_min"] > df_h["ngrc_seed_max"])]
if len(robust):
    print(f"\nQRC robustly beats NG-RC (every seed) from ~{robust['lookahead_tu'].min()} tu "
          f"(~{robust['lookahead_tu'].min()/char_tu:.2f} {'tau_c' if _chaotic else 'periods'}) onward.")
sig = df_h[df_h["ci_lo"] > 0]
if len(sig):
    print(f"QRC significantly beats persistence (pooled CI lo>0) at: {list(sig['lookahead_tu'])} tu")

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.fill_between(df_h["lookahead_tu"], df_h["ci_lo"], df_h["ci_hi"], color="#2a78d6", alpha=0.18, label="QRC pooled 95% CI")
ax.fill_between(df_h["lookahead_tu"], df_h["ngrc_seed_min"], df_h["ngrc_seed_max"], color="#e34948", alpha=0.12, label="NG-RC seed range")
ax.plot(df_h["lookahead_tu"], df_h["skill_qrc"], "o-", color="#2a78d6", label="QRC (mean)")
ax.plot(df_h["lookahead_tu"], df_h["skill_ngrc"], "s--", color="#e34948", label="NG-RC (mean)")
ax.axhline(0.0, color="#898781", lw=1, ls=":", label="persistence")
if _chaotic:
    ax.axvline(char_tu, color="#888", lw=1, ls="-.", alpha=0.6); ax.text(char_tu, 0.92, " tau_c", color="#888", fontsize=9)
ax.set_xscale("log"); ax.set_xlabel("forecast horizon [time units]"); ax.set_ylabel("skill vs persistence")
ax.set_ylim(min(-1.0, df_h["skill_ngrc"].min()), 1.05)
ax.set_title(f"Case {CASE} ({_tag}): skill vs horizon, {len(seed_list)} noise seeds ({qrc.atom_number} atoms, '{HZ_READOUT}')")
ax.grid(True, alpha=0.3); ax.legend(loc="best", fontsize=8); fig.tight_layout()
fig.savefig(ROOT / f"qrc_horizon_ms_{CASE}_{_tag}.png", dpi=160); plt.close(fig)
print(f"Saved: {ROOT / ('qrc_horizon_ms_' + CASE + '_' + _tag + '.png')}")


Regime: chaotic, tau_c ~ 10.3 tu  | seeds: 3  | anchors 60 (train 45/test 15)
Planned emulator runs: ~360  (cap 600)
  emulating seed 1234 (120 runs)...
  emulating seed 1335 (120 runs)...
  emulating seed 1436 (120 runs)...

=== Multi-seed horizon skill (pooled bootstrap CI; per-seed min/max shown) ===
 lookahead_tu  n_seeds  n_test  skill_qrc  ci_lo  ci_hi  qrc_seed_min  qrc_seed_max  skill_ngrc  ngrc_seed_min  ngrc_seed_max  horizon_over_tauc
        1.050        3      15     -1.028 -1.991 -0.156        -2.100        -0.434       0.906          0.901          0.914              0.102
        2.050        3      15     -1.135 -2.137 -0.245        -2.309        -0.486       0.864          0.854          0.876              0.199
        5.150        3      15     -1.397 -2.456 -0.472        -2.870        -0.578       0.592          0.524          0.665              0.500
       10.300        3      15     -1.240 -2.277 -0.327        -2.794        -0.267       0.090         -0.003     

### 4.3 Design sweep — can QRC beat NG-RC and persistence? (emulator, multi-seed, capped)

Sweeps atoms / readout / shots / total_time / probe-times on the emulator and asks, per config, whether the QRC robustly beats both NG-RC and persistence at the long-horizon chaotic regime. **Note the hard limit:** the local emulator only reaches ~12 atoms, while neutral-atom QRC is expected to need tens of atoms — so a null result here characterizes the small-atom regime only and does not settle the hardware-scale question.

In [27]:
# =============================================================================
# CELL E (4.3): EMULATOR design sweep — can QRC beat NG-RC (and persistence) on
# chaotic Case IV by changing atoms / readout / shots / total_time / probe times?
# =============================================================================
# Multi-seed, pooled, capped. For each config it reports skill vs persistence and
# vs NG-RC at the LONG-horizon chaotic regime (a few decorrelation times), plus a
# strict per-seed "robust beat" flag. HARD limit on emulator runs protects the kernel.
#
# HARD LIMIT OF THIS SWEEP: the local emulator stores the full 2^n state vector, so
# it is practical only to ~12-14 atoms. The atom counts where neutral-atom QRC is
# expected to win (tens of atoms) are NOT reachable here -- only on Aquila. So a null
# result here does NOT rule out a hardware-scale advantage; it only characterizes the
# small-atom regime.
# =============================================================================
import json, pickle, hashlib
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge

# ---- sweep setup (small + capped; extend gradually, results cache) ----
SWEEP_SEEDS        = [SEED, SEED + 101]
SWEEP_N_TRAIN      = 40
SWEEP_N_TEST       = 12
SWEEP_WIN_STRIDE_TU = 5.0
SWEEP_BURN_IN_TU   = 500.0
SWEEP_EVAL_FRACS   = [1.0, 2.0, 3.0]      # eval horizons in multiples of decorrelation time
SWEEP_BOOTSTRAP    = 1500
SWEEP_MAX_EMU_RUNS = 1500
SWEEP_MAX_ATOMS    = 12                     # refuse configs above this (emulator safety)

# (label, state_indices, window_length, time_steps, total_time, nshots, readout)
SWEEP_CONFIGS = [
    ("8a_Z_t2_s500",    [0,1,2,3],       2, 2, 4.0, 500,  "Z"),
    ("8a_Z_t2_s1000",   [0,1,2,3],       2, 2, 4.0, 1000, "Z"),
    ("8a_ZZ_t2_s500",   [0,1,2,3],       2, 2, 4.0, 500,  "ZZ"),
    ("8a_Z_t2_T2",      [0,1,2,3],       2, 2, 2.0, 500,  "Z"),
    ("12a_Z_t2_s500",   [0,1,2,3,4,5],   2, 2, 4.0, 500,  "Z"),
    ("8a_Z_t4_s500",    [0,1,2,3],       2, 4, 4.0, 500,  "Z"),
]

_chaotic = (DYNAMICS == "chaotic") or (DYNAMICS == "auto" and str(CASE).upper() == "IV")

def _char_time_tu(x, dt, max_lag_tu=150.0):
    x = np.asarray(x, float) - np.mean(x); n = len(x); ml = min(n - 1, int(round(max_lag_tu / dt)))
    ac = np.correlate(x, x, "full")[n - 1: n - 1 + ml]
    if ac[0] <= 0: return None
    ac = ac / ac[0]; b = np.where(ac < 1.0 / np.e)[0]
    return float(b[0] * dt) if len(b) else None

# clean trajectory (seed-independent) for anchors/targets/timescale
_clean = load_or_generate_dataset(input_npz=None, case=CASE, noise_level=0.0, noise_kind="none",
                                  student_df=STUDENT_DF, seed=SWEEP_SEEDS[0])
XC = np.asarray(_clean["X_sampled"], float)
dt = float(_clean.get("metadata", {}).get("sampling_interval",
                                          np.median(np.diff(np.asarray(_clean["tau_sampled"], float)))))
i0 = int(round(SWEEP_BURN_IN_TU / dt))
char_tu = (_char_time_tu(XC[i0:i0 + min(int(round(600 / dt)), len(XC) - i0), 0], dt) or 10.0) if _chaotic else 75.0
spacing_raw = max(1, int(round(char_tu / dt)))
win_stride_raw = max(1, int(round(SWEEP_WIN_STRIDE_TU / dt)))
H_eval = sorted(set(max(1, int(round(fr * char_tu / dt))) for fr in SWEEP_EVAL_FRACS))
H_max = max(H_eval)

n0 = SWEEP_N_TRAIN + SWEEP_N_TEST
base = i0 + (max(c[2] for c in SWEEP_CONFIGS) - 1) * win_stride_raw
anchors = base + np.arange(n0) * spacing_raw
keep = (anchors + H_max < len(XC)) & (anchors - (max(c[2] for c in SWEEP_CONFIGS) - 1) * win_stride_raw >= 0)
anchors = anchors[keep]; n_anchor = len(anchors)
n_test = min(SWEEP_N_TEST, max(3, n_anchor // 4))
tr_idx = np.arange(0, n_anchor - n_test); te_idx = np.arange(n_anchor - n_test, n_anchor)

cache_path = ROOT / f"qrc_designsweep_emb_{CASE}.pkl"
cache = pickle.load(open(cache_path, "rb")) if cache_path.exists() else {}
def _ek(label, win, tsteps, ttime, nshots, readout, seed):
    return hashlib.md5(json.dumps([CASE, label, win, tsteps, ttime, nshots, readout, int(seed),
        round(char_tu,3), int(spacing_raw), SWEEP_WIN_STRIDE_TU, int(n_anchor)], sort_keys=True).encode()).hexdigest()

# ---- pre-flight budget ----
def _atoms(c): return c[2] * len(c[1])
bad_atoms = [c[0] for c in SWEEP_CONFIGS if _atoms(c) > SWEEP_MAX_ATOMS]
if bad_atoms:
    raise RuntimeError(f"Configs exceed SWEEP_MAX_ATOMS={SWEEP_MAX_ATOMS}: {bad_atoms}. "
                       f"The local emulator can't handle that many atoms; use Aquila for those.")
seed_list = SWEEP_SEEDS if USE_NOISY_INPUT else ["clean"]
planned = 0
for (label, sidx, win, tsteps, ttime, nshots, rdo) in SWEEP_CONFIGS:
    for seed in seed_list:
        if _ek(label, win, tsteps, ttime, nshots, rdo, seed) not in cache:
            planned += n_anchor * tsteps
print(f"Chaotic={_chaotic}, tau_c~{char_tu:.1f} tu | {len(SWEEP_CONFIGS)} configs x {len(seed_list)} seeds")
print(f"Anchors {n_anchor} (train {len(tr_idx)}/test {len(te_idx)}) | eval horizons {[round(h*dt,1) for h in H_eval]} tu")
print(f"Planned emulator runs: ~{planned}  (cap {SWEEP_MAX_EMU_RUNS})")
if planned > SWEEP_MAX_EMU_RUNS:
    raise RuntimeError(f"Refusing ~{planned} Bloqade runs (> {SWEEP_MAX_EMU_RUNS}). Trim SWEEP_CONFIGS / SWEEP_SEEDS.")

_denoise_cache = {}
def _denoised_full(seed):
    if seed in _denoise_cache:
        return _denoise_cache[seed]
    if USE_NOISY_INPUT:
        ds = load_or_generate_dataset(input_npz=None, case=CASE, noise_level=NOISE_LEVEL,
                                      noise_kind=NOISE_KIND, student_df=STUDENT_DF, seed=int(seed))
        src = np.asarray(ds["Y_sampled"], float)
        Xin = np.empty_like(src); _w = 31 if len(src) >= 31 else (len(src)//2*2 - 1)
        for d_ in range(src.shape[1]):
            Xin[:, d_] = savgol_filter(src[:, d_], max(5, _w), 3, mode="interp")
    else:
        Xin = XC
    _denoise_cache[seed] = Xin
    return Xin

def _build_W(seed, sidx, win):
    Xin = _denoised_full(seed)[:, sidx]
    return np.asarray([Xin[a - (win - 1 - np.arange(win)) * win_stride_raw].reshape(-1) for a in anchors])

_rng = np.random.default_rng(0)
def _boot_lo(em, ep, B):
    n = len(em); out = np.empty(B)
    for b in range(B):
        idx = _rng.integers(0, n, n); pp = ep[idx].mean()
        out[b] = 1 - em[idx].mean() / pp if pp > 1e-30 else np.nan
    return np.nanpercentile(out, 2.5)

Xc_all = XC  # for targets per state set
rows = []
for (label, sidx, win, tsteps, ttime, nshots, rdo) in SWEEP_CONFIGS:
    qrc = QRCConfig(atom_number=win * len(sidx), encoding="local", lattice_spacing=QRC_LATTICE_SPACING,
                    encoding_scale=QRC_ENCODING_SCALE, rabi_frequency=QRC_RABI_FREQUENCY,
                    total_time=ttime, time_steps=tsteps, readouts=rdo)
    Xc_sel = Xc_all[:, sidx]
    eq_pool, en_pool, ep_pool = [], [], []
    perseed_q, perseed_n = [], []
    emb_dim = None
    for seed in seed_list:
        W = _build_W(seed, sidx, win)
        ek = _ek(label, win, tsteps, ttime, nshots, rdo, seed)
        if ek in cache:
            E = cache[ek]
        else:
            Wsc = np.clip(MinMaxScaler((0.0, 1.0)).fit(W[tr_idx]).transform(W), 0.0, 1.0)
            E = emulate_qrc_embeddings(qrc, Wsc, nshots=nshots)
            cache[ek] = E; pickle.dump(cache, open(cache_path, "wb"))
        emb_dim = E.shape[1]
        seed_eq, seed_en, seed_ep = [], [], []
        for H in H_eval:
            Y = (Xc_sel[anchors + H] - Xc_sel[anchors]) if PREDICT_DELTA else Xc_sel[anchors + H]
            Yq = Ridge(alpha=1e-4, fit_intercept=True).fit(E[tr_idx], Y[tr_idx]).predict(E[te_idx])
            Yn = build_ngrc_model(degree=2).fit(W[tr_idx], Y[tr_idx]).predict(W[te_idx])
            eq = ((Yq - Y[te_idx])**2).mean(1); en = ((Yn - Y[te_idx])**2).mean(1); ep = (Y[te_idx]**2).mean(1)
            seed_eq.append(eq); seed_en.append(en); seed_ep.append(ep)
        seq = np.concatenate(seed_eq); sen = np.concatenate(seed_en); sep = np.concatenate(seed_ep)
        eq_pool.append(seq); en_pool.append(sen); ep_pool.append(sep)
        perseed_q.append(1 - seq.mean()/sep.mean()); perseed_n.append(1 - sen.mean()/sep.mean())
    EQ = np.concatenate(eq_pool); EN = np.concatenate(en_pool); EP = np.concatenate(ep_pool)
    mp = EP.mean()
    skq = 1 - EQ.mean()/mp; skn = 1 - EN.mean()/mp
    ci_lo = _boot_lo(EQ, EP, SWEEP_BOOTSTRAP)
    rows.append({"config": label, "atoms": win*len(sidx), "readout": rdo, "time_steps": tsteps,
                 "total_time": ttime, "nshots": nshots, "emb_dim": emb_dim,
                 "ok_dim": len(tr_idx) > emb_dim, "skill_qrc": skq, "qrc_ci_lo": ci_lo,
                 "skill_ngrc": skn, "gap_qrc_minus_ngrc": skq - skn,
                 "robust_beat_ngrc": min(perseed_q) > max(perseed_n),
                 "beats_persist": ci_lo > 0})

df = pd.DataFrame(rows).sort_values("gap_qrc_minus_ngrc", ascending=False)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print("\n=== Design sweep @ long-horizon chaotic regime (best QRC-vs-NG-RC gap first) ===")
print(df.to_string(index=False))
df.to_csv(ROOT / f"qrc_designsweep_{CASE}.csv", index=False)

winners = df[(df["gap_qrc_minus_ngrc"] > 0) & (df["robust_beat_ngrc"]) & (df["beats_persist"]) & (df["ok_dim"])]
print("\n=== VERDICT ===")
if len(winners):
    print("Config(s) where QRC robustly beats BOTH NG-RC and persistence (emulator):")
    print(winners[["config","atoms","readout","skill_qrc","skill_ngrc"]].to_string(index=False))
    print("-> worth confirming on Aquila.")
else:
    rb = df[df["robust_beat_ngrc"] & df["ok_dim"]]
    print("No config beats BOTH NG-RC and persistence robustly at <=12 atoms.")
    if len(rb):
        print(f"QRC robustly beats NG-RC (graceful degradation) in: {list(rb['config'])} "
              f"-- but still below persistence (skill<0).")
    print("This characterizes the SMALL-ATOM regime only. The atom counts where neutral-atom")
    print("QRC is expected to win (tens of atoms) are not reachable on the local emulator;")
    print("settling that requires Aquila hardware, not this sweep.")


Chaotic=True, tau_c~10.3 tu | 6 configs x 2 seeds
Anchors 52 (train 40/test 12) | eval horizons [10.3, 20.6, 30.9] tu
Planned emulator runs: ~1456  (cap 1500)

=== Design sweep @ long-horizon chaotic regime (best QRC-vs-NG-RC gap first) ===
       config  atoms readout  time_steps  total_time  nshots  emb_dim  ok_dim  skill_qrc  qrc_ci_lo  skill_ngrc  gap_qrc_minus_ngrc  robust_beat_ngrc  beats_persist
   8a_Z_t2_T2      8       Z           2       2.000     500       16    True      0.190     -0.052      -0.596               0.786              True          False
 8a_Z_t2_s500      8       Z           2       4.000     500       16    True     -0.429     -0.815      -0.596               0.167             False          False
8a_ZZ_t2_s500      8      ZZ           2       4.000     500       72   False     -0.658     -1.413      -0.596              -0.063             False          False
8a_Z_t2_s1000      8       Z           2       4.000    1000       16    True     -0.962     -1.570

### 4.4 `total_time` drill-down — the decisive significance test

The 4.3 sweep flagged reservoir `total_time` as the knob that flipped QRC skill positive (2 us beat the default 4 us). This cell fixes 8 atoms / Z / 2 probe-times and finely varies `total_time` with 3 seeds and a larger test set, asking whether any evolution time makes the QRC **significantly** beat persistence (bootstrap CI lower bound > 0). A positive verdict here is the green light to claim a real advantage and confirm it on Aquila.

In [28]:
# =============================================================================
# CELL F (4.4) v2: TOTAL_TIME drill-down — Camino B, protocolo congelado
# Cambios vs v1 (justificados en la sesión de verificación):
#   1. Readout QRC: RidgeCV (simetría con los clásicos; el alpha=1e-4 fijo estaba
#      sub-regularizado frente al ruido de 500 shots y distorsionaba los valles).
#   2. Menú clásico ampliado: deg1 CV, deg2 CV, deg3 CV y control RFF de dimensión
#      igualada al embedding (features tanh aleatorias + RidgeCV, promedio de 5
#      proyecciones). beats_classical se evalúa contra el MÁXIMO de los cuatro.
#   3. Seis semillas (criterio de robustez per-seed con más poder).
# Nota de encuadre: el barrido de total_time es un corte 1D de la estructura
# temporal de los probes (ver mapa de pares); se reporta como caracterización.
# =============================================================================
import json, pickle, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import RidgeCV

# ---- fixed reservoir config (the 8-atom Z baseline) ----
TT_STATE_INDICES = [0, 1, 2, 3]
TT_WINDOW        = 2
TT_TIME_STEPS    = 2
TT_READOUT       = "Z"
TT_NSHOTS        = 500
# ---- the axis we drill ----
TT_VALUES        = [1.0, 1.5, 2.0, 2.5, 3.0]   # total_time (us); all <= 4 us (Aquila-realizable)
# ---- evaluation (v2: 6 seeds) ----
TT_SEEDS         = [SEED, SEED + 101, SEED + 202, SEED + 303, SEED + 404, SEED + 505]
TT_N_TRAIN       = 44
TT_N_TEST        = 22
TT_WIN_STRIDE_TU = 5.0
TT_BURN_IN_TU    = 500.0
TT_EVAL_FRACS    = [1.0, 2.0, 3.0]      # multiples of decorrelation time
TT_BOOTSTRAP     = 3000
TT_MAX_EMU_RUNS  = 4000                 # subido: 6 semillas x 5 tt puede requerir ~3960 en frío
TT_ALPHAS        = np.logspace(-6, 3, 19)
TT_RFF_REPS      = 5                    # proyecciones RFF promediadas

_chaotic = (DYNAMICS == "chaotic") or (DYNAMICS == "auto" and str(CASE).upper() == "IV")

def _char_time_tu(x, dt, max_lag_tu=150.0):
    x = np.asarray(x, float) - np.mean(x); n = len(x); ml = min(n - 1, int(round(max_lag_tu / dt)))
    ac = np.correlate(x, x, "full")[n - 1: n - 1 + ml]
    if ac[0] <= 0: return None
    ac = ac / ac[0]; b = np.where(ac < 1.0 / np.e)[0]
    return float(b[0] * dt) if len(b) else None

_clean = load_or_generate_dataset(input_npz=None, case=CASE, noise_level=0.0, noise_kind="none",
                                  student_df=STUDENT_DF, seed=TT_SEEDS[0])
XC = np.asarray(_clean["X_sampled"], float)
dt = float(_clean.get("metadata", {}).get("sampling_interval",
                                          np.median(np.diff(np.asarray(_clean["tau_sampled"], float)))))
i0 = int(round(TT_BURN_IN_TU / dt))
char_tu = (_char_time_tu(XC[i0:i0 + min(int(round(600/dt)), len(XC)-i0), 0], dt) or 10.0) if _chaotic else 75.0
spacing_raw = max(1, int(round(char_tu / dt)))
win_stride_raw = max(1, int(round(TT_WIN_STRIDE_TU / dt)))
H_eval = sorted(set(max(1, int(round(fr * char_tu / dt))) for fr in TT_EVAL_FRACS))
H_max = max(H_eval)

n0 = TT_N_TRAIN + TT_N_TEST
base = i0 + (TT_WINDOW - 1) * win_stride_raw
anchors = base + np.arange(n0) * spacing_raw
keep = (anchors + H_max < len(XC)) & (anchors - (TT_WINDOW - 1) * win_stride_raw >= 0)
anchors = anchors[keep]; n_anchor = len(anchors)
n_test = min(TT_N_TEST, max(3, n_anchor // 3))
tr_idx = np.arange(0, n_anchor - n_test); te_idx = np.arange(n_anchor - n_test, n_anchor)

Xc_sel = XC[:, TT_STATE_INDICES]
seed_list = TT_SEEDS if USE_NOISY_INPUT else ["clean"]

cache_path = ROOT / f"qrc_totaltime_emb_{CASE}.pkl"
cache = pickle.load(open(cache_path, "rb")) if cache_path.exists() else {}
def _ek(tt, seed):
    return hashlib.md5(json.dumps([CASE, "TT", tt, TT_STATE_INDICES, TT_WINDOW, TT_TIME_STEPS, TT_READOUT,
        TT_NSHOTS, int(seed), round(char_tu,3), int(spacing_raw), TT_WIN_STRIDE_TU, int(n_anchor)],
        sort_keys=True).encode()).hexdigest()

planned = sum(n_anchor * TT_TIME_STEPS for tt in TT_VALUES for seed in seed_list if _ek(tt, seed) not in cache)
print(f"chaotic={_chaotic}, tau_c~{char_tu:.1f} tu | total_time values {TT_VALUES} | seeds {len(seed_list)}")
print(f"anchors {n_anchor} (train {len(tr_idx)}/test {len(te_idx)}) | eval horizons {[round(h*dt,1) for h in H_eval]} tu")
print(f"pooled test points per total_time: {len(seed_list)*len(te_idx)*len(H_eval)}")
print(f"Planned emulator runs: ~{planned}  (cap {TT_MAX_EMU_RUNS})")
if planned > TT_MAX_EMU_RUNS:
    raise RuntimeError(f"Refusing ~{planned} runs (> {TT_MAX_EMU_RUNS}); fewer TT_VALUES/TT_SEEDS or lower TT_N_TRAIN/TEST.")

_dn = {}
def _denoised(seed):
    if seed in _dn: return _dn[seed]
    if USE_NOISY_INPUT:
        ds = load_or_generate_dataset(input_npz=None, case=CASE, noise_level=NOISE_LEVEL,
                                      noise_kind=NOISE_KIND, student_df=STUDENT_DF, seed=int(seed))
        src = np.asarray(ds["Y_sampled"], float); X = np.empty_like(src)
        _w = 31 if len(src) >= 31 else (len(src)//2*2 - 1)
        for d_ in range(src.shape[1]): X[:, d_] = savgol_filter(src[:, d_], max(5, _w), 3, mode="interp")
    else:
        X = XC
    _dn[seed] = X; return X

def _W(seed):
    Xin = _denoised(seed)[:, TT_STATE_INDICES]
    return np.asarray([Xin[a - (TT_WINDOW - 1 - np.arange(TT_WINDOW)) * win_stride_raw].reshape(-1) for a in anchors])

_rng = np.random.default_rng(0)
def _boot(em, ep, B):
    n = len(em); out = np.empty(B)
    for b in range(B):
        idx = _rng.integers(0, n, n); pp = ep[idx].mean()
        out[b] = 1 - em[idx].mean()/pp if pp > 1e-30 else np.nan
    return np.nanpercentile(out, 2.5), np.nanpercentile(out, 97.5)

# ---- v2: control RFF de dimensión igualada (features tanh aleatorias + RidgeCV) ----
class _RFF:
    def __init__(self, dim, in_dim, rng, scale=1.0):
        self.G = rng.normal(0, scale/np.sqrt(in_dim), (in_dim, dim))
        self.b = rng.uniform(-np.pi, np.pi, dim)
    def transform(self, X):
        return np.tanh(X @ self.G + self.b)

def _ridgecv():
    return RidgeCV(alphas=TT_ALPHAS, fit_intercept=True)

W_by_seed = {s: _W(s) for s in seed_list}
EMB_DIM = TT_WINDOW * len(TT_STATE_INDICES) * TT_TIME_STEPS if TT_READOUT == "Z" else None
rows = []
for tt in TT_VALUES:
    qrc = QRCConfig(atom_number=TT_WINDOW*len(TT_STATE_INDICES), encoding="local",
                    lattice_spacing=QRC_LATTICE_SPACING, encoding_scale=QRC_ENCODING_SCALE,
                    rabi_frequency=QRC_RABI_FREQUENCY, total_time=tt, time_steps=TT_TIME_STEPS, readouts=TT_READOUT)
    eq_pool, ep_pool = [], []
    ecls_pool = {m: [] for m in ("d1", "d2", "d3", "rff")}
    psq, pscls_best = [], []
    for seed in seed_list:
        W = W_by_seed[seed]; ek = _ek(tt, seed)
        if ek in cache:
            E = cache[ek]
        else:
            Wsc = np.clip(MinMaxScaler((0.0, 1.0)).fit(W[tr_idx]).transform(W), 0.0, 1.0)
            E = emulate_qrc_embeddings(qrc, Wsc, nshots=TT_NSHOTS)
            cache[ek] = E; pickle.dump(cache, open(cache_path, "wb"))
        rff_dim = E.shape[1]  # dimensión igualada al embedding QRC real
        seq, sep = [], []
        secls = {m: [] for m in ecls_pool}
        for H in H_eval:
            Y = (Xc_sel[anchors + H] - Xc_sel[anchors]) if PREDICT_DELTA else Xc_sel[anchors + H]
            # v2: readout QRC con RidgeCV (simetría)
            Yq  = _ridgecv().fit(E[tr_idx], Y[tr_idx]).predict(E[te_idx])
            seq.append(((Yq-Y[te_idx])**2).mean(1))
            # menú clásico: NG-RC deg1/deg2/deg3 (todos CV) sobre la MISMA ventana
            for m, deg in (("d1",1), ("d2",2), ("d3",3)):
                Yc = build_ngrc_model(degree=deg).fit(W[tr_idx], Y[tr_idx]).predict(W[te_idx])
                secls[m].append(((Yc-Y[te_idx])**2).mean(1))
            # control RFF de dimensión igualada, promedio de TT_RFF_REPS proyecciones
            sc = StandardScaler().fit(W[tr_idx])
            Wtr, Wte = sc.transform(W[tr_idx]), sc.transform(W[te_idx])
            mses = []
            for r in range(TT_RFF_REPS):
                rff = _RFF(rff_dim, W.shape[1], np.random.default_rng(100 + r))
                Yr = _ridgecv().fit(rff.transform(Wtr), Y[tr_idx]).predict(rff.transform(Wte))
                mses.append(((Yr - Y[te_idx])**2).mean(1))
            secls["rff"].append(np.mean(mses, axis=0))
            sep.append((Y[te_idx]**2).mean(1))
        seq = np.concatenate(seq); sep = np.concatenate(sep)
        eq_pool.append(seq); ep_pool.append(sep)
        for m in ecls_pool: ecls_pool[m].append(np.concatenate(secls[m]))
        psq.append(1 - seq.mean()/sep.mean())
        pscls_best.append(max(1 - np.concatenate(secls[m]).mean()/sep.mean() for m in secls))
    EQ = np.concatenate(eq_pool); EP = np.concatenate(ep_pool); mp = EP.mean()
    skq = 1 - EQ.mean()/mp
    sk_cls = {m: 1 - np.concatenate(ecls_pool[m]).mean()/mp for m in ecls_pool}
    skn_best = max(sk_cls.values())
    lo, hi = _boot(EQ, EP, TT_BOOTSTRAP)
    rows.append({"total_time": tt, "skill_qrc": skq, "ci_lo": lo, "ci_hi": hi,
                 "qrc_seed_min": min(psq), "qrc_seed_max": max(psq),
                 "skill_ngrc_lin": sk_cls["d1"], "skill_ngrc": sk_cls["d2"],
                 "skill_ngrc_d3": sk_cls["d3"], "skill_rff": sk_cls["rff"],
                 "skill_cls_best": skn_best,
                 "beats_persist_sig": lo > 0,
                 "beats_classical": lo > skn_best,
                 "beats_classical_robust": min(psq) > max(pscls_best)})

df = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print("\n=== total_time drill-down v2 (8 atoms, Z; readout CV; pooled 95% CI; 6 seeds) ===")
print(df.to_string(index=False))
df.to_csv(ROOT / f"qrc_totaltime_{CASE}_v2.csv", index=False)

# --------------------------- VERDICT (Camino B) ------------------------------
print("\n=== VERDICT (Camino B: characterization, NOT advantage) ===")
qrc_best = df.sort_values("skill_qrc", ascending=False).iloc[0]
cls_best = df["skill_cls_best"].max()
print(f"Classical menu (total_time-independent): lin={df['skill_ngrc_lin'].iloc[0]:.3f}, "
      f"deg2={df['skill_ngrc'].iloc[0]:.3f}, deg3={df['skill_ngrc_d3'].iloc[0]:.3f}, "
      f"RFF(dim-matched)={df['skill_rff'].iloc[0]:.3f} -> best={cls_best:.3f}")
print(f"QRC best: {qrc_best['skill_qrc']:.3f} at total_time={qrc_best['total_time']} us "
      f"(CI [{qrc_best['ci_lo']:.3f}, {qrc_best['ci_hi']:.3f}]).")
print("Note: the total_time sweep is a 1D slice of the probe-time structure "
      "(information decay + narrow anti-resonances + multi-probe complementarity); "
      "report as characterization.")
if df["beats_classical"].any():
    w = df[df["beats_classical"]]
    print(f"CAUTION: QRC CI-lower-bound exceeds the best classical at {list(w['total_time'])} us.")
    print("  Before any claim: check beats_classical_robust, add seeds, and account for "
          "multiplicity across every configuration explored in the project.")
else:
    print("At 8 atoms the QRC does NOT exceed the best fair classical baseline. "
          "This is the expected outcome at this scale (cf. Kornjaca et al.) and is the "
          "declared no-advantage result of the paper.")

chaotic=True, tau_c~10.3 tu | total_time values [1.0, 1.5, 2.0, 2.5, 3.0] | seeds 6
anchors 66 (train 44/test 22) | eval horizons [10.3, 20.6, 30.9] tu
pooled test points per total_time: 396
Planned emulator runs: ~3960  (cap 4000)

=== total_time drill-down v2 (8 atoms, Z; readout CV; pooled 95% CI; 6 seeds) ===
 total_time  skill_qrc  ci_lo  ci_hi  qrc_seed_min  qrc_seed_max  skill_ngrc_lin  skill_ngrc  skill_ngrc_d3  skill_rff  skill_cls_best  beats_persist_sig  beats_classical  beats_classical_robust
      1.000      0.698  0.659  0.733         0.655         0.756           0.714       0.371         -1.691      0.580           0.714               True            False                   False
      1.500     -0.197 -0.511  0.046        -1.004         0.006           0.714       0.371         -1.691      0.580           0.714              False            False                   False
      2.000      0.658  0.578  0.722         0.378         0.759           0.714       0.371        

## 4.5 Aquila hardware run (Camino B) — submit 5-point `total_time` scan

Submits the resonance scan `[0.8, 1.0, 1.2, 1.5, 2.0]` us over **two dataset seeds**, single probe time (`Z`, `time_steps=1`) to keep the run at **360 tasks (~$288 @ 50 shots)**. This is a **viability + shape** run, not a significance test: the goal is to acquire the resonant curve on real Aquila and check it **reproduces the emulator shape** and to **measure** the decoherence suppression. Run once with the gate OFF to see the exact budget.

**Free dry-run first:** set `USE_HARDWARE=True`, `USE_MOCK=True`, `HW_CONFIRM_SCAN=True` (`quera.mock`, no cost) to validate submit+fetch+analysis end-to-end before paying.

In [ ]:
# =============================================================================
# CELL G (4.5): AQUILA HARDWARE RUN (Camino B) — SUBMIT 5-point total_time scan
#               over TWO noise-realization seeds, single probe time (Z).
# Goal of this run is NOT a significance verdict. It is a VIABILITY + SHAPE run:
# acquire the resonant total_time curve on real Aquila over two dataset seeds so
# the analysis cell (4.6) can report (i) hardware-emulator shape agreement,
# (ii) a MEASURED decoherence suppression r(t) with effective T2, and (iii) the
# curve with error bars. Reuses submit_local_tasks_hardware(...) (your native path).
# Budget: n_anchor x time_steps x n_tt x n_seeds tasks. Gate must be set to submit.
# =============================================================================
import json
import numpy as np
from scipy.signal import savgol_filter
from sklearn.preprocessing import MinMaxScaler

# ---- fixed reservoir config (8-atom Z baseline) ----
HW_STATE_INDICES = [0, 1, 2, 3]
HW_WINDOW        = 2
HW_TIME_STEPS    = 1          # single probe time -> emb_dim 8; keeps the run at 360 tasks
HW_READOUT       = "Z"
# ---- scan + scope (Camino B: resonance SHAPE, peak/shoulders/valley/echo) ----
HW_TT_SCAN       = [0.8, 1.0, 1.2, 1.5, 2.0]   # us; 1.0 peak, 1.5 valley, 2.0 echo, 0.8/1.2 frame the peak
HW_NSHOTS        = 50
HW_SEEDS         = [SEED, SEED + 101]          # TWO dataset realizations (reproducibility, not significance)
HW_N_TRAIN       = 24
HW_N_TEST        = 12                           # n0 = 36 anchors (train 24 > emb_dim 8)
HW_WIN_STRIDE_TU = 5.0
HW_BURN_IN_TU    = 500.0
HW_EVAL_FRACS    = [1.0, 2.0, 3.0]              # multiples of decorrelation time
# ---- SAFETY GATE: must be True to actually submit to real Aquila ----
HW_CONFIRM_SCAN  = True

_chaotic = (DYNAMICS == "chaotic") or (DYNAMICS == "auto" and str(CASE).upper() == "IV")

def _char_time_tu(x, dt, max_lag_tu=150.0):
    x = np.asarray(x, float) - np.mean(x); n = len(x); ml = min(n - 1, int(round(max_lag_tu / dt)))
    ac = np.correlate(x, x, "full")[n - 1: n - 1 + ml]
    if ac[0] <= 0: return None
    ac = ac / ac[0]; b = np.where(ac < 1.0 / np.e)[0]
    return float(b[0] * dt) if len(b) else None

# clean trajectory (targets/timescale; seed-independent) -> defines anchors once
_clean = load_or_generate_dataset(input_npz=None, case=CASE, noise_level=0.0, noise_kind="none",
                                  student_df=STUDENT_DF, seed=HW_SEEDS[0])
XC = np.asarray(_clean["X_sampled"], float)
dt = float(_clean.get("metadata", {}).get("sampling_interval",
                                          np.median(np.diff(np.asarray(_clean["tau_sampled"], float)))))
i0 = int(round(HW_BURN_IN_TU / dt))
char_tu = (_char_time_tu(XC[i0:i0 + min(int(round(600/dt)), len(XC)-i0), 0], dt) or 10.0) if _chaotic else 75.0
spacing_raw   = max(1, int(round(char_tu / dt)))
win_stride_raw = max(1, int(round(HW_WIN_STRIDE_TU / dt)))
H_eval = sorted(set(max(1, int(round(fr * char_tu / dt))) for fr in HW_EVAL_FRACS))
H_max = max(H_eval)

n0 = HW_N_TRAIN + HW_N_TEST
base = i0 + (HW_WINDOW - 1) * win_stride_raw
anchors = base + np.arange(n0) * spacing_raw
keep = (anchors + H_max < len(XC)) & (anchors - (HW_WINDOW - 1) * win_stride_raw >= 0)
anchors = anchors[keep]; n_anchor = len(anchors)
n_test = min(HW_N_TEST, max(3, n_anchor // 3))
tr_idx = np.arange(0, n_anchor - n_test); te_idx = np.arange(n_anchor - n_test, n_anchor)
emb_dim = HW_WINDOW * len(HW_STATE_INDICES) * HW_TIME_STEPS  # Z, time_steps=1 -> 8

# ---- build per-seed windows (raw + scaled). Anchors identical across seeds. ----
def _windows_for_seed(seed):
    if USE_NOISY_INPUT:
        ds = load_or_generate_dataset(input_npz=None, case=CASE, noise_level=NOISE_LEVEL,
                                      noise_kind=NOISE_KIND, student_df=STUDENT_DF, seed=int(seed))
        src = np.asarray(ds["Y_sampled"], float)[:, HW_STATE_INDICES]
        Xin = np.empty_like(src); _w = 31 if len(src) >= 31 else (len(src)//2*2 - 1)
        for d_ in range(src.shape[1]):
            Xin[:, d_] = savgol_filter(src[:, d_], max(5, _w), 3, mode="interp")
    else:
        Xin = XC[:, HW_STATE_INDICES]
    Wr = np.asarray([Xin[a - (HW_WINDOW - 1 - np.arange(HW_WINDOW)) * win_stride_raw].reshape(-1) for a in anchors])
    Wsc = np.clip(MinMaxScaler((0.0, 1.0)).fit(Wr[tr_idx]).transform(Wr), 0.0, 1.0)
    return Wr, Wsc

W_by_seed, Wsc_by_seed = {}, {}
for sd in HW_SEEDS:
    Wr, Wsc = _windows_for_seed(sd)
    W_by_seed[int(sd)] = Wr; Wsc_by_seed[int(sd)] = Wsc

# ---- BUDGET ----
tasks_per_tt_seed = n_anchor * HW_TIME_STEPS
tasks_total = tasks_per_tt_seed * len(HW_TT_SCAN) * len(HW_SEEDS)
usd = tasks_total * (0.30 + 0.01 * HW_NSHOTS)
print("=" * 68)
print("AQUILA HARDWARE RUN BUDGET (Camino B: 5-point scan x 2 seeds)")
print("=" * 68)
print(f"  config: 8 atoms, '{HW_READOUT}', {HW_TIME_STEPS} probe time -> emb_dim {emb_dim}")
print(f"  total_time values: {HW_TT_SCAN} us")
print(f"  noise seeds: {HW_SEEDS}  ({len(HW_SEEDS)} dataset realizations)")
print(f"  anchors {n_anchor} (train {len(tr_idx)} > emb_dim {emb_dim}? {len(tr_idx) > emb_dim}; test {len(te_idx)})")
print(f"  tasks: {n_anchor} x {HW_TIME_STEPS} probe x {len(HW_TT_SCAN)} tt x {len(HW_SEEDS)} seeds = {tasks_total}")
print(f"  shots/task: {HW_NSHOTS}")
print(f"  EST. COST: {tasks_total} x (0.30 + 0.01x{HW_NSHOTS}) = ${usd:,.0f}")
print("=" * 68)

# save meta so 4.6 uses identical anchors/windows/targets/seeds
meta_path = ROOT / f"ttscan_meta_{CASE}.npz"
np.savez(meta_path, anchors=anchors, tr_idx=tr_idx, te_idx=te_idx,
         H_eval=np.array(H_eval), tt_scan=np.array(HW_TT_SCAN), seeds=np.array(HW_SEEDS, dtype=int),
         W_stack=np.stack([W_by_seed[int(s)] for s in HW_SEEDS]),
         Wsc_stack=np.stack([Wsc_by_seed[int(s)] for s in HW_SEEDS]),
         dt=dt, char_tu=char_tu, state_indices=np.array(HW_STATE_INDICES),
         window=HW_WINDOW, time_steps=HW_TIME_STEPS, nshots=HW_NSHOTS)
print(f"Saved scan metadata: {meta_path}")

# ---- GATE ----
_real     = bool(USE_HARDWARE) and (not USE_MOCK) and bool(CONFIRM_REAL_QPU_SUBMISSION) and bool(HW_CONFIRM_SCAN)
_mock_dry = bool(USE_HARDWARE) and bool(USE_MOCK) and bool(HW_CONFIRM_SCAN)   # free quera.mock end-to-end test
_ready = _real or _mock_dry
if not _ready:
    print("\nNOT SUBMITTING (budget preview only).")
    print("  - REAL Aquila run: USE_HARDWARE=True, USE_MOCK=False, "
          "CONFIRM_REAL_QPU_SUBMISSION=True, HW_CONFIRM_SCAN=True.")
    print("  - FREE end-to-end test first (recommended): USE_HARDWARE=True, USE_MOCK=True, "
          "HW_CONFIRM_SCAN=True (quera.mock, no cost) -- also creates the prefixes file so 4.6 runs.")
    print(f"  current: USE_HARDWARE={USE_HARDWARE}, USE_MOCK={USE_MOCK}, "
          f"CONFIRM_REAL_QPU_SUBMISSION={CONFIRM_REAL_QPU_SUBMISSION}, HW_CONFIRM_SCAN={HW_CONFIRM_SCAN}")
else:
    _mode = "REAL AQUILA ($ charged)" if _real else "MOCK (quera.mock, free)"
    print(f"\nSUBMITTING {tasks_total} tasks ({len(HW_TT_SCAN)} tt x {len(HW_SEEDS)} seeds) -- mode: {_mode}")
    submitted = {}
    for sd in HW_SEEDS:
        Wsc = Wsc_by_seed[int(sd)]
        for tt in HW_TT_SCAN:
            qrc = QRCConfig(atom_number=HW_WINDOW*len(HW_STATE_INDICES), encoding="local",
                            lattice_spacing=QRC_LATTICE_SPACING, encoding_scale=QRC_ENCODING_SCALE,
                            rabi_frequency=QRC_RABI_FREQUENCY, total_time=float(tt),
                            time_steps=HW_TIME_STEPS, readouts=HW_READOUT)
            prefix = f"ttscan_{CASE}_s{int(sd)}_tt{str(tt).replace('.','p')}"
            print(f"\n--- seed {int(sd)}, total_time = {tt} us  (prefix {prefix}) ---")
            paths = submit_local_tasks_hardware(qrc, Wsc, prefix=prefix, nshots=HW_NSHOTS, use_mock=USE_MOCK)
            submitted[f"{tt}|{int(sd)}"] = prefix
            print(f"  submitted {len(paths)} tasks")
    json.dump(submitted, open(ROOT / f"ttscan_prefixes_{CASE}.json", "w"))
    print(f"\nAll tasks submitted. Prefixes saved. Run the FETCH cell once tasks complete.")


## 4.6 Aquila hardware run (Camino B) — fetch + characterization

Run after 4.5's tasks complete. **No go/no-go significance verdict.** Reports, from measured data: **(1)** hardware↔emulator shape agreement of the `skill(total_time)` curve (Pearson/Spearman, sign-pattern match); **(2)** a **measured** suppression `r(t)=skill_hw/skill_emu` fitted as `A·exp(-t/T2_eff)` — replacing the projected `0.88·exp(-t/4us)` placeholder with a fit; **(3)** the skill curve **with error bars**, saved as CSV/PNG/LaTeX to replace the `_PROJECTED` artifacts. The matched emulator reference is recomputed here at the same config (`time_steps=1`) so `r(t)` isolates decoherence, not embedding size (free local Bloqade, cached).  Also emits **(4)** an emulator-vs-hardware **embedding-agreement heatmap** at the resonant-peak `total_time` (mirrors the QuEra/Kornjača timeseries demo), as direct viability evidence that the hardware reproduces the `\langle Z_i\rangle` embeddings.

In [ ]:
# =============================================================================
# CELL H (4.6): AQUILA HARDWARE RUN (Camino B) — FETCH + CHARACTERIZATION
# Run AFTER 4.5's tasks complete. NO significance/go-no-go verdict. Instead it
# reports, from MEASURED hardware data over the 5-point scan x 2 seeds:
#   (1) hardware<->emulator SHAPE agreement of the skill(total_time) curve
#       (Pearson, Spearman, sign-pattern match),
#   (2) a MEASURED decoherence suppression r(t)=skill_hw/skill_emu, fitted as
#       A*exp(-t/T2_eff) -> reports A (=1-eps_read) and the effective T2,
#       replacing the *projected* r(t)=0.88*exp(-t/4us) placeholder with a fit,
#   (3) the skill curve WITH ERROR BARS (bootstrap CI over pooled seeds/horizons),
#       saved as CSV + PNG + LaTeX to REPLACE the _PROJECTED artifacts.
# The matched emulator reference is recomputed here at the SAME config
# (time_steps=1) so r(t) isolates decoherence, not embedding dimensionality.
# =============================================================================
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from scipy.stats import pearsonr, spearmanr
from scipy.optimize import curve_fit

HW_BOOTSTRAP   = 3000
EMU_FIT_THRESH = 0.10     # only fit r(t) where the emulator skill is robustly positive
EMU_REF_MAX    = 800      # guard: refuse runaway local Bloqade reference compute

meta = np.load(ROOT / f"ttscan_meta_{CASE}.npz")
anchors = meta["anchors"]; tr_idx = meta["tr_idx"]; te_idx = meta["te_idx"]
H_eval = [int(h) for h in meta["H_eval"]]; tt_scan = [float(t) for t in meta["tt_scan"]]
seeds = [int(s) for s in meta["seeds"]]; dt = float(meta["dt"]); char_tu = float(meta["char_tu"])
sidx = [int(i) for i in meta["state_indices"]]; HW_WINDOW = int(meta["window"])
HW_TIME_STEPS = int(meta["time_steps"]); HW_NSHOTS = int(meta["nshots"])
W_stack = meta["W_stack"]; Wsc_stack = meta["Wsc_stack"]   # (n_seeds, n_anchor, .)
seed_pos = {s: k for k, s in enumerate(seeds)}

_pref_path = ROOT / f"ttscan_prefixes_{CASE}.json"
if not _pref_path.exists():
    raise RuntimeError(
        "No submitted tasks found (ttscan_prefixes_" + str(CASE) + ".json missing).\n"
        "Cell 4.6 only FETCHES what 4.5 SUBMITTED. Run 4.5 with submission enabled first:\n"
        "  - FREE test: USE_HARDWARE=True, USE_MOCK=True, HW_CONFIRM_SCAN=True (quera.mock), then re-run 4.6.\n"
        "  - REAL run:  USE_HARDWARE=True, USE_MOCK=False, CONFIRM_REAL_QPU_SUBMISSION=True, HW_CONFIRM_SCAN=True,\n"
        "    wait for Aquila tasks to COMPLETE, then run 4.6.")
prefixes = json.load(open(_pref_path))

# clean targets (seed-independent)
_clean = load_or_generate_dataset(input_npz=None, case=CASE, noise_level=0.0, noise_kind="none",
                                  student_df=STUDENT_DF, seed=seeds[0])
Xc_sel = np.asarray(_clean["X_sampled"], float)[:, sidx]

_rng = np.random.default_rng(0)
def _skill_and_ci(EQ, EP, B=HW_BOOTSTRAP):
    mp = EP.mean(); sk = 1 - EQ.mean()/mp if mp > 1e-30 else np.nan
    n = len(EQ); out = np.empty(B)
    for b in range(B):
        idx = _rng.integers(0, n, n); pp = EP[idx].mean()
        out[b] = 1 - EQ[idx].mean()/pp if pp > 1e-30 else np.nan
    return sk, np.nanpercentile(out, 2.5), np.nanpercentile(out, 97.5)

def _pooled_errors(embed_of_seed):
    """embed_of_seed: dict seed-> embeddings (n_anchor, emb_dim). Returns pooled EQ, EP."""
    eq_all, ep_all = [], []
    for sd in seeds:
        E = embed_of_seed[sd]
        for H in H_eval:
            Y = (Xc_sel[anchors + H] - Xc_sel[anchors]) if PREDICT_DELTA else Xc_sel[anchors + H]
            Yq = Ridge(alpha=1e-4, fit_intercept=True).fit(E[tr_idx], Y[tr_idx]).predict(E[te_idx])
            eq_all.append(((Yq - Y[te_idx])**2).mean(1)); ep_all.append((Y[te_idx]**2).mean(1))
    return np.concatenate(eq_all), np.concatenate(ep_all)

# ---------- (A) HARDWARE embeddings (fetch) ----------
print("Fetching hardware embeddings ...")
hw_emb = {}   # tt -> {seed -> E}
for tt in tt_scan:
    per_seed = {}
    for sd in seeds:
        key = f"{tt}|{sd}"
        if key not in prefixes:
            print(f"  [warn] no prefix for tt={tt}, seed={sd}; skipping that cell"); continue
        qrc = QRCConfig(atom_number=HW_WINDOW*len(sidx), encoding="local", lattice_spacing=QRC_LATTICE_SPACING,
                        encoding_scale=QRC_ENCODING_SCALE, rabi_frequency=QRC_RABI_FREQUENCY,
                        total_time=float(tt), time_steps=HW_TIME_STEPS, readouts="Z")
        per_seed[sd] = np.asarray(fetch_local_embeddings_hardware(qrc, prefixes[key], count=len(anchors)), float)
    if len(per_seed) == len(seeds):
        hw_emb[tt] = per_seed
    else:
        print(f"  [warn] tt={tt} incomplete across seeds; excluded from analysis")

# ---------- (B) MATCHED EMULATOR reference at the SAME config (cached) ----------
emu_cache_path = ROOT / f"ttscan_emu_ref_{CASE}.npz"
emu_emb = {}
if emu_cache_path.exists():
    z = np.load(emu_cache_path)
    for tt in tt_scan:
        if all(f"{tt}|{sd}" in z.files for sd in seeds):
            emu_emb[tt] = {sd: z[f"{tt}|{sd}"] for sd in seeds}
todo = [tt for tt in hw_emb if tt not in emu_emb]
planned = len(todo) * len(seeds) * len(anchors) * HW_TIME_STEPS
if planned > EMU_REF_MAX:
    raise RuntimeError(f"Emulator reference would need ~{planned} Bloqade runs (> {EMU_REF_MAX}); "
                       f"reduce HW_TT_SCAN/seeds/anchors.")
if todo:
    print(f"Computing matched emulator reference (local Bloqade): ~{planned} runs over {len(todo)} tt x {len(seeds)} seeds ...")
    store = dict(z) if emu_cache_path.exists() else {}
    for tt in todo:
        for sd in seeds:
            qrc = QRCConfig(atom_number=HW_WINDOW*len(sidx), encoding="local", lattice_spacing=QRC_LATTICE_SPACING,
                            encoding_scale=QRC_ENCODING_SCALE, rabi_frequency=QRC_RABI_FREQUENCY,
                            total_time=float(tt), time_steps=HW_TIME_STEPS, readouts="Z")
            Wsc = Wsc_stack[seed_pos[sd]]
            E = np.asarray(emulate_qrc_embeddings(qrc, Wsc, nshots=HW_NSHOTS), float)
            store[f"{tt}|{sd}"] = E
            emu_emb.setdefault(tt, {})[sd] = E
        print(f"  emulator tt={tt} done")
    np.savez(emu_cache_path, **store)
    print(f"  cached emulator reference -> {emu_cache_path}")

# ---------- (C) skill curves (hardware + emulator), pooled over 2 seeds ----------
rows = []
for tt in tt_scan:
    if tt not in hw_emb or tt not in emu_emb:
        continue
    EQh, EPh = _pooled_errors(hw_emb[tt]); skh, lo, hi = _skill_and_ci(EQh, EPh)
    EQe, EPe = _pooled_errors(emu_emb[tt]); ske = 1 - EQe.mean()/EPe.mean()
    rows.append(dict(total_time=tt, skill_hw=skh, ci_lo=lo, ci_hi=hi, skill_emu=ske,
                     n_pooled=len(EQh)))
df = pd.DataFrame(rows).sort_values("total_time").reset_index(drop=True)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print("\n=== Aquila hardware vs matched emulator (skill vs persistence; 2 seeds pooled) ===")
print(df[["total_time", "skill_hw", "ci_lo", "ci_hi", "skill_emu", "n_pooled"]].to_string(index=False))
df.to_csv(ROOT / f"qrc_totaltime_hw_{CASE}.csv", index=False)

# ---------- (1) SHAPE AGREEMENT ----------
print("\n=== (1) Hardware<->emulator SHAPE agreement of the total_time curve ===")
if len(df) >= 3:
    pr, pp = pearsonr(df["skill_hw"], df["skill_emu"])
    sr, sp = spearmanr(df["skill_hw"], df["skill_emu"])
    sign_match = int(np.sum(np.sign(df["skill_hw"]) == np.sign(df["skill_emu"])))
    print(f"  Pearson r = {pr:.3f} (p={pp:.3f}),  Spearman rho = {sr:.3f} (p={sp:.3f}) over {len(df)} total_time points")
    print(f"  sign-pattern match: {sign_match}/{len(df)} points share the emulator sign "
          f"(peak>0 at integer Rabi periods, dip<0 near half-period)")
    print("  NOTE: with ~5 points this quantifies SHAPE reproduction, it is not a significance test.")
else:
    print("  too few completed total_time points for a correlation.")

# ---------- (2) MEASURED suppression r(t) = skill_hw / skill_emu ----------
print("\n=== (2) Measured decoherence suppression r(t) = skill_hw / skill_emu ===")
df["r_t"] = np.where(np.abs(df["skill_emu"]) > 1e-6, df["skill_hw"]/df["skill_emu"], np.nan)
# Fit r(t) ONLY where the suppression is physically meaningful: the emulator is robustly
# positive, the hardware STILL retains positive skill, and the ratio is in (0, 1.5].
# Negative/blown-up ratios mean the hardware lost the structure at that point -- that is a
# finding to report descriptively, not data to feed an exp-decay fit.
fit_mask = ((df["skill_emu"] > EMU_FIT_THRESH) & (df["skill_hw"] > 0)
            & np.isfinite(df["r_t"]) & (df["r_t"] > 0) & (df["r_t"] <= 1.5))
tf = df.loc[fit_mask, "total_time"].to_numpy(float); rf = df.loc[fit_mask, "r_t"].to_numpy(float)
print(df[["total_time", "skill_emu", "skill_hw", "r_t"]].to_string(index=False))
print(f"  (r(t) fit uses total_time where skill_emu>{EMU_FIT_THRESH} AND skill_hw>0 AND 0<r<=1.5: {list(tf)} us)")
A_fit = T2_fit = None
if len(tf) >= 3:
    def _model(t, A, T2): return A * np.exp(-t / T2)
    try:
        popt, pcov = curve_fit(_model, tf, rf, p0=[0.85, 4.0],
                               bounds=([0.0, 0.3], [1.5, 50.0]), maxfev=20000)
        perr = np.sqrt(np.diag(pcov)); A_, T2_ = popt
        # reject boundary-pinned / hugely-uncertain fits as not meaningful
        pinned = (A_ < 0.02 or T2_ <= 0.32 or T2_ >= 49.0
                  or not np.all(np.isfinite(perr)) or perr[1] > 5*max(T2_, 1e-6))
        if pinned:
            print("  r(t) fit hit a parameter bound / huge uncertainty -> NOT a meaningful suppression law.")
            print("  Interpretation: hardware did not retain the resonant structure cleanly enough to define")
            print("  an exp-decay r(t). Report the pointwise ratios and the surviving point(s) instead.")
        else:
            A_fit, T2_fit = A_, T2_
            print(f"  FIT  r(t) = A * exp(-t/T2):  A = {A_fit:.3f} +/- {perr[0]:.3f}  (=> readout/SPAM retention 1-eps)")
            print(f"                               T2_eff = {T2_fit:.2f} +/- {perr[1]:.2f} us  (effective analog coherence)")
            print(f"  Compare to the PROJECTED placeholder (A=0.88, T2=4.0 us): "
                  f"A {'higher' if A_fit>0.88 else 'lower'}, T2 {'longer' if T2_fit>4 else 'shorter'} than assumed.")
    except Exception as e:
        print(f"  r(t) fit did not converge ({e}); reporting pointwise ratios only.")
else:
    print(f"  only {len(tf)} point(s) keep positive hardware skill where the emulator is positive -- not enough")
    print("  to fit r(t). That paucity is itself the result: decoherence erased the structure at the rest.")

# ---------- (3) FIGURE with error bars (replaces _PROJECTED) ----------
fig, ax = plt.subplots(figsize=(7.6, 4.5))
yerr = np.vstack([df["skill_hw"] - df["ci_lo"], df["ci_hi"] - df["skill_hw"]])
ax.axhline(0.0, color="#333", lw=1.2, ls=":", label="persistence baseline")
ax.plot(df["total_time"], df["skill_emu"], "s--", color="#1f77b4", alpha=0.85, label="emulator (matched config)")
ax.errorbar(df["total_time"], df["skill_hw"], yerr=yerr, fmt="o-", color="#7a3fb5",
            capsize=4, lw=1.8, label="Aquila (measured, 2 seeds)")
if A_fit is not None:
    tg = np.linspace(min(tt_scan), max(tt_scan), 60)
    ax.plot(tg, A_fit*np.exp(-tg/T2_fit)*np.interp(tg, df["total_time"], df["skill_emu"]),
            "-", color="#d62728", alpha=0.7, lw=1.4,
            label=f"fit: emu x {A_fit:.2f}exp(-t/{T2_fit:.1f}us)")
ax.set_xlabel("reservoir total_time [us]"); ax.set_ylabel("skill vs persistence")
ax.set_title(f"Case {CASE}: Aquila resonant total_time curve vs emulator (8 atoms, Z, {HW_NSHOTS} shots/seed)")
ax.grid(True, alpha=0.3); ax.legend(loc="best", fontsize=9); fig.tight_layout()
_figp = ROOT / f"qrc_totaltime_hw_{CASE}.png"; fig.savefig(_figp, dpi=160); plt.close(fig)
print(f"\nSaved measured figure: {_figp}")

# ---------- (4) EMBEDDING AGREEMENT heatmap (mirrors Kornjaca et al. Fig. 2a) ----------
# Direct viability evidence: hardware reproduces the emulator <Z_i> embeddings (plus noise),
# shown at the resonant-peak total_time for the test windows -- the same emulator-vs-experiment
# embedding comparison used in the QuEra timeseries demo.
peak_tt = min(df["total_time"], key=lambda t: abs(t - 1.0)) if len(df) else None
if peak_tt is not None and peak_tt in hw_emb and peak_tt in emu_emb:
    sd0 = seeds[0]
    Eh = np.asarray(hw_emb[peak_tt][sd0])[te_idx]    # (n_test, emb_dim) = <Z_i>
    Ee = np.asarray(emu_emb[peak_tt][sd0])[te_idx]
    comp_corr = [np.corrcoef(Ee[:, j], Eh[:, j])[0, 1]
                 for j in range(Ee.shape[1]) if np.std(Ee[:, j]) > 1e-9 and np.std(Eh[:, j]) > 1e-9]
    mean_comp_corr = float(np.nanmean(comp_corr)) if comp_corr else np.nan
    rel_fro = float(np.linalg.norm(Eh - Ee) / (np.linalg.norm(Ee) + 1e-12))
    print(f"\n=== (4) Embedding agreement at peak total_time={peak_tt} us (seed {sd0}) ===")
    print(f"  mean per-component corr(emulator, hardware) over test windows = {mean_comp_corr:.3f}")
    print(f"  relative Frobenius difference ||E_hw - E_emu|| / ||E_emu|| = {rel_fro:.3f}")
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.6), sharey=True)
    vmin = float(min(Ee.min(), Eh.min())); vmax = float(max(Ee.max(), Eh.max()))
    im = None
    for ax_, M_, ttl in [(axes[0], Ee.T, "emulator"), (axes[1], Eh.T, "Aquila (measured)")]:
        im = ax_.imshow(M_, aspect="auto", vmin=vmin, vmax=vmax, cmap="viridis")
        ax_.set_xlabel("test window"); ax_.set_title(ttl)
    axes[0].set_ylabel(r"embedding component $\langle Z_i\rangle$")
    fig.colorbar(im, ax=axes, fraction=0.025)
    fig.suptitle(f"Case {CASE}: QRC embeddings, emulator vs hardware (total_time={peak_tt} us, seed {sd0})")
    _ehp = ROOT / f"qrc_embedding_agreement_{CASE}.png"; fig.savefig(_ehp, dpi=160, bbox_inches="tight"); plt.close(fig)
    print(f"  Saved embedding-agreement figure: {_ehp}")

# LaTeX table (measured) to replace table_hw_projected.tex
def _f(x): return "--" if (x is None or (isinstance(x, float) and not np.isfinite(x))) else f"{x:.3f}"
lt = [r"\begin{table}[htbp]", r"\caption{Measured Aquila hardware skill versus matched emulator reference for the "
      r"reservoir evolution-time scan (Case " + str(CASE) + r", 8 atoms, $Z$ readout, " + str(HW_NSHOTS) +
      r" shots, two dataset seeds). Skill is relative to the persistence baseline; "
      r"$r(t)=\mathrm{skill}_{\mathrm{hw}}/\mathrm{skill}_{\mathrm{emu}}$.}",
      r"\label{tab:hw_measured}", r"\centering",
      r"\begin{tabular}{cccccc}", r"\hline",
      r"$t_{\mathrm{tot}}$ [$\mu$s] & skill$_{\mathrm{hw}}$ & 95\% CI & skill$_{\mathrm{emu}}$ & $r(t)$ \\", r"\hline"]
for _, r in df.iterrows():
    lt.append(f"{r['total_time']:.1f} & {_f(r['skill_hw'])} & [{_f(r['ci_lo'])}, {_f(r['ci_hi'])}] "
              f"& {_f(r['skill_emu'])} & {_f(r['r_t'])} \\\\")
lt += [r"\hline", r"\end{tabular}"]
if A_fit is not None:
    lt.append(r"\\[2pt] \footnotesize Fitted suppression $r(t)=A\,e^{-t/T_2}$ with "
              f"$A={A_fit:.2f}$, $T_2={T2_fit:.1f}\\,\\mu$s.")
lt.append(r"\end{table}")
_texp = ROOT / f"table_hw_measured_{CASE}.tex"; open(_texp, "w").write("\n".join(lt))
print(f"Saved measured LaTeX table: {_texp}")
print("\nThese MEASURED artifacts replace qrc_totaltime_hw_*_PROJECTED.csv / table_hw_projected.tex /")
print("fig_totaltime_emu_vs_hw_projected.png in the manuscript once you have run the real Aquila scan.")


## Notes

### Safe first real Aquila run

Use:

```python
USE_HARDWARE = True
USE_MOCK = False
CONFIRM_REAL_QPU_SUBMISSION = True

CASE = "II"
ENCODING = "local"
QRC_READOUT = "Z"
MAX_TRAIN_SAMPLES = 10
MAX_TEST_SAMPLES = 5
NSHOTS = 30
```

This submits:

- local encoding: one task per data point
- 10 train + 5 test = 15 tasks
- 30 shots/task = 450 requested shots

### Recommended readout regularization

For hardware embeddings, start with:

```python
QRC_READOUT_ALPHA = 1e-4
```

Then reuse the saved embeddings to compare:

```python
QRC_READOUT_ALPHA = 1e-5
QRC_READOUT_ALPHA = 1e-4
QRC_READOUT_ALPHA = 1e-3
```

Changing this alpha does not require re-running Aquila if the embeddings are already fetched.

### Local vs global

For first hardware verification, use `ENCODING="local"`.  
The global variant is supported but submits `N_samples * QRC_TIME_STEPS` tasks and is more sensitive to pulse construction details.

In [ ]:
from pathlib import Path
R = Path.cwd(); CASE = "IV"; PFX = "aquila_IV_local_Z_w2"
expected = [
    f"ttscan_meta_{CASE}.npz",
    f"ttscan_prefixes_{CASE}.json",
    f"ttscan_emu_ref_{CASE}.npz",
    f"qrc_totaltime_hw_{CASE}.csv",
    f"qrc_totaltime_hw_{CASE}.png",
    f"qrc_embedding_agreement_{CASE}.png",
    f"table_hw_measured_{CASE}.tex",
]
for f in expected:
    print(f"{'OK ' if (R/f).exists() else 'MISSING':7} {f}")
for d in (f"{PFX}_tasks", f"{PFX}_results"):
    p = R/d; n = len(list(p.glob('*.json'))) if p.exists() else 0
    print(f"{'OK ' if n==360 else 'CHECK':7} {d}/  -> {n} json (expect 360)")

In [ ]:
import json, re
from pathlib import Path
R=Path.cwd(); CASE="IV"; PFX="aquila_IV_local_Z_w2"
pref=json.load(open(R/f"ttscan_prefixes_{CASE}.json"))
valid_prefixes=set(pref.values())
print("prefijos del run actual:", len(pref), "->", len(valid_prefixes), "unicos (espera 10)")
tdir=R/f"{PFX}_tasks"
files=list(tdir.glob("*.json"))
def base(fn): return re.sub(r"_\d+$","",fn.stem)   # quita _{idx}
current=[f for f in files if base(f) in valid_prefixes]
stale=[f for f in files if base(f) not in valid_prefixes]
print(f"tareas del run actual: {len(current)} (espera 360)")
print(f"tareas viejas (sobrantes): {len(stale)}")
print("prefijos viejos:", sorted({base(f) for f in stale}))

In [ ]:
# =========================
# Backup critical artifacts
# =========================

import shutil
from pathlib import Path

artifact_dir = Path("/home/ec2-user/amazon-braket-examples/examples/analog_hamiltonian_simulation/aquila_II_local_Z_w2_tasks")

zip_base = artifact_dir.parent / "aquila_II_local_Z_w2_tasks"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=artifact_dir)

print("Created backup ZIP:", zip_path)

In [29]:
!python3 lcurve_fixedtest.py run 400

DONE
